# 07B1 — From Customer Orders to Efficient Deliveries
## Open/public traffic data + driver soft overtime + VRP + 3D packing

**XtremePython 2026 demo — open/public road data + workforce-aware routing + physical loading**

This notebook models:

- 20 mock customers in Irvine
- a **40-SKU hard-coded product master**
- an explicit **customer order table**
- each order line contains **SKU + quantity**
- physical boxes are generated from ordered quantities
- **2 Large Trucks + 3 Small Trucks (5 trucks total)**
- customer time windows
- **driver working shifts with soft overtime optimization**
- **driver-to-truck qualification constraints**
- a pre-dispatch **truck availability filter** based on staffing
- vehicle payload constraints
- aggregate volume constraints
- actual cargo-bed dimensions
- **3D bin-packing feasibility**
- iterative feedback from packing back into vehicle routing
- OpenStreetMap / OSMnx road network
- **City of Irvine official Speed + ADT open GIS data**
- **Caltrans CWWP2 District 12 current highway travel times**
- traffic-aware edge travel times and customer-to-customer travel-time matrix
- final route, overtime, and packing visualization

### Data flow

```text
OSM road network
    +
City of Irvine official Speed / ADT
    +
Caltrans CWWP2 current highway travel time
            ↓
Open/public traffic-aware edge travel times
            ↓
Product Master + Customer Orders
            ↓
Physical shipment boxes
            ↓
Customer total weight / aggregate volume
            ↓
Driver qualification + scheduled shifts
            ↓
Soft overtime penalty when a route returns late
            ↓
VRP
            ↓
3D Packing check for each assigned truck
            ↓
All FIT? ── YES ──> Accept
   │
   NO
   ↓
Add packing conflict cut
   ↓
Re-run VRP
```

### Optimization policy

The objective combines routing efficiency with workforce scheduling:

> Minimize **traffic-aware road travel time** while strongly minimizing **total driver overtime**.

The scheduled shift end is soft: overtime is allowed when needed, but every overtime minute is penalized.

A route is accepted only when every assigned truck load is physically packable:

> **3D packing does not need to be optimal, but NOT FIT is unacceptable.**


### Version note

This **07B1** notebook is the open/public-data counterpart of **07A1**.

The workforce and optimization logic is kept the same as 07A1:

- driver roster
- driver-to-truck qualification
- hard driver shift start
- **soft driver shift end**
- overtime allowed but strongly penalized
- customer time windows
- payload / cargo-volume constraints
- iterative 3D-packing feedback cuts

The main change is the **road / traffic data layer**.

```text
07A1
OSM + TomTom commercial real-time traffic
            ↓
same VRP / driver / packing logic

07B1
OSM + City of Irvine GIS + Caltrans CWWP2
            ↓
same VRP / driver / packing logic
```

### Open-data interpretation

- **City of Irvine Speed** supplies official local/arterial baseline speeds where available.
- **City of Irvine ADT 2022** is retained as traffic-volume context; it is **not** treated as live speed.
- **Caltrans CWWP2 District 12** supplies current highway travel-time observations.
- Dynamic current-traffic coverage is therefore more limited than TomTom and depends on the public data available in the region.

This is intentional: the goal is to show that a useful traffic-aware routing model may still be assembled from open/public GIS and transportation data, depending on the local government and transportation agencies.


# Chapter 1 — Same Base Delivery Scenario as 07A1

**Presentation role:** Traffic-data-layer difference demo.

Customers, products, orders, trucks, drivers, soft overtime, VRP, and 3D packing are intentionally kept aligned with 07A1.

Do **not** re-explain these common sections in detail.

## 1. Install packages

In [ ]:
!pip -q install "osmnx>=2.1,<3" ortools folium py3dbp


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Imports and basic helpers

In [ ]:
import math
import requests
from collections import defaultdict
from datetime import datetime, timezone
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
import networkx as nx
import folium
from IPython.display import display

import osmnx as ox
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
from py3dbp import Packer, Bin, Item

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Customer source time windows in this demo are expressed as
# minutes from 08:00 (legacy input format).
CUSTOMER_DAY_START_HOUR = 8

# The optimization clock starts earlier so driver shifts such as
# 06:00-12:00 can be represented naturally.
SCHEDULE_ZERO_HOUR = 6
SCHEDULE_END_HOUR = 18
SCHEDULE_HORIZON_MIN = (
    SCHEDULE_END_HOUR - SCHEDULE_ZERO_HOUR
) * 60

CUSTOMER_TO_SCHEDULE_OFFSET_MIN = (
    CUSTOMER_DAY_START_HOUR - SCHEDULE_ZERO_HOUR
) * 60


def customer_hhmm(minutes_from_8am: int) -> str:
    """Display original customer-window minutes (0 = 08:00)."""
    total = (
        CUSTOMER_DAY_START_HOUR * 60
        + int(minutes_from_8am)
    )
    return f"{total // 60:02d}:{total % 60:02d}"


def hhmm(minutes_from_schedule_zero: int) -> str:
    """Display optimizer time (0 = 06:00)."""
    total = (
        SCHEDULE_ZERO_HOUR * 60
        + int(minutes_from_schedule_zero)
    )
    return f"{total // 60:02d}:{total % 60:02d}"


def clock_to_schedule_min(hhmm_text: str) -> int:
    """Convert 'HH:MM' to optimizer minutes from 06:00."""
    hour, minute = map(int, hhmm_text.split(":"))
    value = hour * 60 + minute - SCHEDULE_ZERO_HOUR * 60
    if value < 0 or value > SCHEDULE_HORIZON_MIN:
        raise ValueError(
            f"Time {hhmm_text} is outside the "
            f"{SCHEDULE_ZERO_HOUR:02d}:00-"
            f"{SCHEDULE_END_HOUR:02d}:00 planning horizon."
        )
    return value


def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def build_haversine_matrix(df):
    n = len(df)
    matrix = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            matrix[i, j] = haversine_km(
                df.loc[i, "lat"], df.loc[i, "lon"],
                df.loc[j, "lat"], df.loc[j, "lon"]
            )
    return matrix

## 3. Customer locations, time windows, and service times

In [ ]:
# Index 0 is the warehouse/depot.
# The 20 customer coordinates are fixed for a reproducible live demo.

locations = [
    ("DEPOT", "Distribution Center, Irvine, CA", 33.6846, -117.8265, 0,   0,   0),
    ("C01", "Mock Address 01, Irvine, CA", 33.6695, -117.8231, 0,   540, 12),
    ("C02", "Mock Address 02, Irvine, CA", 33.6500, -117.8420, 0,   540, 10),
    ("C03", "Mock Address 03, Irvine, CA", 33.6980, -117.7980, 240, 480, 12),
    ("C04", "Mock Address 04, Irvine, CA", 33.7050, -117.7740, 0,   540, 10),
    ("C05", "Mock Address 05, Irvine, CA", 33.7210, -117.7900, 60,  240, 10),
    ("C06", "Mock Address 06, Irvine, CA", 33.7170, -117.8310, 0,   540, 15),
    ("C07", "Mock Address 07, Irvine, CA", 33.7030, -117.8600, 180, 360, 10),
    ("C08", "Mock Address 08, Irvine, CA", 33.6880, -117.8650, 0,   540, 12),
    ("C09", "Mock Address 09, Irvine, CA", 33.6770, -117.8570, 240, 480, 10),
    ("C10", "Mock Address 10, Irvine, CA", 33.6650, -117.8580, 0,   540, 10),
    ("C11", "Mock Address 11, Irvine, CA", 33.6400, -117.8330, 0,   180, 15),
    ("C12", "Mock Address 12, Irvine, CA", 33.6470, -117.8070, 0,   540, 10),
    ("C13", "Mock Address 13, Irvine, CA", 33.6610, -117.7900, 300, 540, 12),
    ("C14", "Mock Address 14, Irvine, CA", 33.6750, -117.7850, 0,   540, 10),
    ("C15", "Mock Address 15, Irvine, CA", 33.6900, -117.7870, 120, 300, 15),
    ("C16", "Mock Address 16, Irvine, CA", 33.7140, -117.8080, 0,   540, 10),
    ("C17", "Mock Address 17, Irvine, CA", 33.7310, -117.8110, 240, 420, 12),
    ("C18", "Mock Address 18, Irvine, CA", 33.7360, -117.8450, 0,   540, 10),
    ("C19", "Mock Address 19, Irvine, CA", 33.7110, -117.8850, 60,  300, 12),
    ("C20", "Mock Address 20, Irvine, CA", 33.6800, -117.8930, 300, 540, 10),
]

customer_df = pd.DataFrame(
    locations,
    columns=[
        "customer_id", "address", "lat", "lon",
        "window_start", "window_end", "service_min"
    ]
)


customer_names = {
    f"C{i:02d}": f"Customer {chr(64+i)}"
    for i in range(1, 21)
}
customer_names["DEPOT"] = "Depot / Warehouse"
customer_df["customer_name"] = customer_df["customer_id"].map(customer_names)

customer_df["time_window"] = customer_df.apply(
    lambda r: f"{customer_hhmm(r.window_start)}-{customer_hhmm(r.window_end)}",
    axis=1
)

display(customer_df)

,customer_id,address,lat,lon,window_start,window_end,service_min,customer_name,time_window
0,DEPOT,"Distribution Center, Irvine, CA",33.6846,-117.8265,0,0,0,Depot / Warehouse,08:00-08:00
1,C01,"Mock Address 01, Irvine, CA",33.6695,-117.8231,0,540,12,Customer A,08:00-17:00
2,C02,"Mock Address 02, Irvine, CA",33.6500,-117.8420,0,540,10,Customer B,08:00-17:00
3,C03,"Mock Address 03, Irvine, CA",33.6980,-117.7980,240,480,12,Customer C,12:00-16:00
4,C04,"Mock Address 04, Irvine, CA",33.7050,-117.7740,0,540,10,Customer D,08:00-17:00
5,C05,"Mock Address 05, Irvine, CA",33.7210,-117.7900,60,240,10,Customer E,09:00-12:00
6,C06,"Mock Address 06, Irvine, CA",33.7170,-117.8310,0,540,15,Customer F,08:00-17:00
7,C07,"Mock Address 07, Irvine, CA",33.7030,-117.8600,180,360,10,Customer G,11:00-14:00
8,C08,"Mock Address 08, Irvine, CA",33.6880,-117.8650,0,540,12,Customer H,08:00-17:00
9,C09,"Mock Address 09, Irvine, CA",33.6770,-117.8570,240,480,10,Customer I,12:00-16:00


## 4. Hard-coded product master — 40 SKUs

The product master is independent of customers.

Each SKU defines:

- SKU / product name
- shape
- Length × Width × Height
- unit weight
- unit volume

All products in this version are **rectangular cuboid boxes**.

Some SKUs are intentionally longer than the Small Truck's 300 cm cargo-bed length.  
A customer ordering one of those SKUs may pass a simple aggregate-volume test but still fail the physical 3D Packing check.

In [ ]:
# Units:
#   dimensions = cm
#   weight = kg
#
# This is the PRODUCT MASTER.
# No customer assignment is stored here.

products = [
    ("SKU001","Standard Carton S","BOX", 60, 40, 35, 12),
    ("SKU002","Long Equipment A","BOX",310, 50, 40, 36),  # too long for Small Truck
    ("SKU003","Standard Carton M","BOX", 80, 55, 45, 18),
    ("SKU004","Standard Carton L","BOX",110, 70, 60, 28),
    ("SKU005","Bulk Carton A","BOX",100, 80, 65, 32),
    ("SKU006","Long Equipment B","BOX",320, 55, 45, 42),  # too long for Small Truck
    ("SKU007","Machine Case A","BOX",120, 75, 70, 35),
    ("SKU008","Parts Carton A","BOX", 85, 60, 55, 16),
    ("SKU009","Parts Carton B","BOX", 95, 70, 60, 22),
    ("SKU010","Long Equipment C","BOX",325, 45, 40, 30),  # too long for Small Truck
    ("SKU011","Machine Case B","BOX",130, 80, 70, 38),
    ("SKU012","Parts Carton C","BOX", 90, 60, 50, 20),
    ("SKU013","Bulk Carton B","BOX",105, 75, 65, 26),
    ("SKU014","Long Equipment D","BOX",330, 50, 45, 44),  # too long for Small Truck
    ("SKU015","Machine Case C","BOX",115, 75, 60, 30),
    ("SKU016","Standard Carton XS","BOX", 55, 40, 35, 10),
    ("SKU017","Bulk Carton C","BOX",100, 70, 60, 24),
    ("SKU018","Parts Carton D","BOX", 75, 55, 50, 15),
    ("SKU019","Machine Case D","BOX",125, 80, 65, 34),
    ("SKU020","Long Equipment E","BOX",315, 50, 45, 40),  # too long for Small Truck
    ("SKU021","Bulk Carton D","BOX",105, 75, 70, 27),
    ("SKU022","Parts Carton E","BOX", 80, 55, 45, 16),
    ("SKU023","Machine Case E","BOX",115, 75, 65, 31),
    ("SKU024","Long Equipment F","BOX",340, 50, 40, 46),  # too long for Small Truck
    ("SKU025","Bulk Carton E","BOX",100, 80, 70, 29),
    ("SKU026","Parts Carton F","BOX", 85, 60, 50, 18),
    ("SKU027","Machine Case F","BOX",120, 75, 65, 33),
    ("SKU028","Standard Carton XXS","BOX", 50, 35, 30,  8),
    ("SKU029","Bulk Carton F","BOX",110, 70, 60, 25),
    ("SKU030","Parts Carton G","BOX", 90, 65, 55, 19),
    ("SKU031","Machine Case G","BOX",130, 80, 70, 37),
    ("SKU032","Parts Carton H","BOX", 85, 60, 50, 17),
    ("SKU033","Bulk Carton G","BOX",105, 75, 65, 26),
    ("SKU034","Long Equipment G","BOX",315, 60, 45, 43),  # too long for Small Truck
    ("SKU035","Machine Case H","BOX",115, 80, 70, 32),
    ("SKU036","Standard Carton S2","BOX", 65, 45, 40, 13),
    ("SKU037","Machine Case I","BOX",125, 75, 65, 34),
    ("SKU038","Parts Carton I","BOX", 90, 65, 55, 18),
    ("SKU039","Bulk Carton H","BOX",100, 70, 60, 23),
    ("SKU040","Long Equipment H","BOX",335, 45, 40, 39),  # too long for Small Truck
]

product_df = pd.DataFrame(
    products,
    columns=[
        "sku", "product_name", "shape",
        "length_cm", "width_cm", "height_cm", "unit_weight_kg"
    ]
)

product_df["unit_volume_cm3"] = (
    product_df["length_cm"]
    * product_df["width_cm"]
    * product_df["height_cm"]
)
product_df["unit_volume_liters"] = product_df["unit_volume_cm3"] / 1000.0

display(product_df)

print("SKUs in product master:", len(product_df))

,sku,product_name,shape,length_cm,width_cm,height_cm,unit_weight_kg,unit_volume_cm3,unit_volume_liters
0,SKU001,Standard Carton S,BOX,60,40,35,12,84000,84.000
1,SKU002,Long Equipment A,BOX,310,50,40,36,620000,620.000
2,SKU003,Standard Carton M,BOX,80,55,45,18,198000,198.000
3,SKU004,Standard Carton L,BOX,110,70,60,28,462000,462.000
4,SKU005,Bulk Carton A,BOX,100,80,65,32,520000,520.000
5,SKU006,Long Equipment B,BOX,320,55,45,42,792000,792.000
6,SKU007,Machine Case A,BOX,120,75,70,35,630000,630.000
7,SKU008,Parts Carton A,BOX,85,60,55,16,280500,280.500
8,SKU009,Parts Carton B,BOX,95,70,60,22,399000,399.000
9,SKU010,Long Equipment C,BOX,325,45,40,30,585000,585.000


SKUs in product master: 40


## 5. Customer order table — who ordered what, and how many?

This is the missing business layer between customers and physical cargo.

Each line explicitly defines:

- `order_id`
- `customer_id`
- `sku`
- `quantity`

For example:

```text
Customer A
  SKU001 × 2
  SKU002 × 1
```

The routing model does **not** route SKUs directly.  
It routes customers after aggregating the weight and volume of all ordered units for each customer.

The packing model then expands every ordered quantity back into individual physical boxes.

In [ ]:
# Hard-coded ORDER LINES.
# Each customer has two SKU lines, with explicit quantities.

orders = [
    ("ORD-C01","C01","SKU001",2), ("ORD-C01","C01","SKU002",1),
    ("ORD-C02","C02","SKU003",3), ("ORD-C02","C02","SKU004",1),
    ("ORD-C03","C03","SKU005",2), ("ORD-C03","C03","SKU006",1),
    ("ORD-C04","C04","SKU007",1), ("ORD-C04","C04","SKU008",3),
    ("ORD-C05","C05","SKU009",2), ("ORD-C05","C05","SKU010",1),
    ("ORD-C06","C06","SKU011",1), ("ORD-C06","C06","SKU012",3),
    ("ORD-C07","C07","SKU013",2), ("ORD-C07","C07","SKU014",1),
    ("ORD-C08","C08","SKU015",1), ("ORD-C08","C08","SKU016",4),
    ("ORD-C09","C09","SKU017",2), ("ORD-C09","C09","SKU018",3),
    ("ORD-C10","C10","SKU019",1), ("ORD-C10","C10","SKU020",1),
    ("ORD-C11","C11","SKU021",2), ("ORD-C11","C11","SKU022",3),
    ("ORD-C12","C12","SKU023",1), ("ORD-C12","C12","SKU024",1),
    ("ORD-C13","C13","SKU025",2), ("ORD-C13","C13","SKU026",2),
    ("ORD-C14","C14","SKU027",1), ("ORD-C14","C14","SKU028",4),
    ("ORD-C15","C15","SKU029",2), ("ORD-C15","C15","SKU030",3),
    ("ORD-C16","C16","SKU031",1), ("ORD-C16","C16","SKU032",3),
    ("ORD-C17","C17","SKU033",2), ("ORD-C17","C17","SKU034",1),
    ("ORD-C18","C18","SKU035",1), ("ORD-C18","C18","SKU036",4),
    ("ORD-C19","C19","SKU037",1), ("ORD-C19","C19","SKU038",3),
    ("ORD-C20","C20","SKU039",2), ("ORD-C20","C20","SKU040",1),
]

order_df = pd.DataFrame(
    orders,
    columns=["order_id", "customer_id", "sku", "quantity"]
)

# Attach customer name and product attributes.
order_detail_df = (
    order_df
    .merge(
        customer_df[["customer_id", "customer_name"]],
        on="customer_id",
        how="left"
    )
    .merge(
        product_df,
        on="sku",
        how="left"
    )
)

order_detail_df["line_weight_kg"] = (
    order_detail_df["quantity"]
    * order_detail_df["unit_weight_kg"]
)

order_detail_df["line_volume_liters"] = (
    order_detail_df["quantity"]
    * order_detail_df["unit_volume_liters"]
)

display(order_detail_df[[
    "order_id", "customer_id", "customer_name",
    "sku", "product_name", "quantity",
    "length_cm", "width_cm", "height_cm",
    "unit_weight_kg", "line_weight_kg",
    "line_volume_liters"
]])

print("Order lines:", len(order_df))
print("Physical units ordered:", int(order_df["quantity"].sum()))

,order_id,customer_id,customer_name,sku,product_name,quantity,length_cm,width_cm,height_cm,unit_weight_kg,line_weight_kg,line_volume_liters
0,ORD-C01,C01,Customer A,SKU001,Standard Carton S,2,60,40,35,12,24,168.000
1,ORD-C01,C01,Customer A,SKU002,Long Equipment A,1,310,50,40,36,36,620.000
2,ORD-C02,C02,Customer B,SKU003,Standard Carton M,3,80,55,45,18,54,594.000
3,ORD-C02,C02,Customer B,SKU004,Standard Carton L,1,110,70,60,28,28,462.000
4,ORD-C03,C03,Customer C,SKU005,Bulk Carton A,2,100,80,65,32,64,1040.000
5,ORD-C03,C03,Customer C,SKU006,Long Equipment B,1,320,55,45,42,42,792.000
6,ORD-C04,C04,Customer D,SKU007,Machine Case A,1,120,75,70,35,35,630.000
7,ORD-C04,C04,Customer D,SKU008,Parts Carton A,3,85,60,55,16,48,841.500
8,ORD-C05,C05,Customer E,SKU009,Parts Carton B,2,95,70,60,22,44,798.000
9,ORD-C05,C05,Customer E,SKU010,Long Equipment C,1,325,45,40,30,30,585.000


Order lines: 40
Physical units ordered: 76


## 6. Truck fleet — two Large Trucks and three Small Trucks

In [ ]:
# Truck dimensions are INTERNAL cargo-bed dimensions.
# This is demo data, not a specification for a particular real truck model.
#
# Fleet:
#   Truck 1-2 = LARGE
#   Truck 3-5 = SMALL
#
# All five trucks are AVAILABLE to the optimizer.
# A truck may remain unused if the routing solution does not need it.

trucks = [
    # vehicle_id, truck_name, type, length, width, height, max_payload
    (0, "Truck 1", "LARGE", 450, 220, 220, 1000),
    (1, "Truck 2", "LARGE", 450, 220, 220, 1000),
    (2, "Truck 3", "SMALL", 300, 170, 170,  500),
    (3, "Truck 4", "SMALL", 300, 170, 170,  500),
    (4, "Truck 5", "SMALL", 300, 170, 170,  500),
]

truck_df = pd.DataFrame(
    trucks,
    columns=[
        "vehicle_id", "truck_name", "truck_type",
        "length_cm", "width_cm", "height_cm", "max_payload_kg"
    ]
)

truck_df["cargo_volume_liters"] = (
    truck_df["length_cm"]
    * truck_df["width_cm"]
    * truck_df["height_cm"]
    / 1000.0
)
truck_df["cargo_volume_m3"] = truck_df["cargo_volume_liters"] / 1000.0

display(truck_df)

print("Fleet size:", len(truck_df), "trucks")
print(
    "Large / Small:",
    int((truck_df["truck_type"] == "LARGE").sum()),
    "/",
    int((truck_df["truck_type"] == "SMALL").sum())
)
print("Fleet payload capacity:", truck_df["max_payload_kg"].sum(), "kg")
print(
    "Fleet geometric volume:",
    round(truck_df["cargo_volume_m3"].sum(), 2),
    "m³"
)

,vehicle_id,truck_name,truck_type,length_cm,width_cm,height_cm,max_payload_kg,cargo_volume_liters,cargo_volume_m3
0,0,Truck 1,LARGE,450,220,220,1000,21780.0,21.78
1,1,Truck 2,LARGE,450,220,220,1000,21780.0,21.78
2,2,Truck 3,SMALL,300,170,170,500,8670.0,8.67
3,3,Truck 4,SMALL,300,170,170,500,8670.0,8.67
4,4,Truck 5,SMALL,300,170,170,500,8670.0,8.67


Fleet size: 5 trucks
Large / Small: 2 / 3
Fleet payload capacity: 3500 kg
Fleet geometric volume: 69.57 m³


## Driver roster and working-shift constraints

The trucks and drivers are modeled as separate business inputs.

For this  demo, each truck is assigned one qualified driver before optimization.

The route optimizer treats the **start of the driver shift as a hard constraint**, but the scheduled shift end is a **soft constraint**:

> Returning after the scheduled shift end is allowed, but every overtime minute is penalized.

This lets the model produce a feasible delivery plan instead of failing immediately when a small amount of overtime is unavoidable.

The final report shows overtime by driver so a dispatcher can consider:

- swapping shifts,
- reassigning a route,
- asking another qualified driver to cover,
- or accepting the overtime when it is operationally reasonable.

> This version models one continuous shift per driver. Breaks and split shifts remain possible extensions.


In [ ]:
# ============================================================
# DRIVER ROSTER + TRUCK QUALIFICATIONS
# ============================================================
#
# Scenario for the  demo:
#
#   Driver A  FULL_TIME  06:00-15:00  SMALL + LARGE
#   Driver B  FULL_TIME  07:00-16:00  SMALL + LARGE
#   Driver C  FULL_TIME  08:00-17:00  SMALL + LARGE
#   Driver D  PART_TIME  07:00-12:00  SMALL only
#   Driver E  PART_TIME  13:00-18:00  SMALL only
#
# The dispatcher first determines which physical trucks can be
# staffed by qualified drivers. The VRP is solved only after those
# truck operating windows are fixed.

drivers = [
    # driver_id, driver_name, employment, shift_start, shift_end,
    # can_small, can_large
    ("D01", "Driver A", "FULL_TIME", "06:00", "15:00", True,  True),
    ("D02", "Driver B", "FULL_TIME", "07:00", "16:00", True,  True),
    ("D03", "Driver C", "FULL_TIME", "08:00", "17:00", True,  True),
    ("D04", "Driver D", "PART_TIME", "07:00", "12:00", True,  False),
    ("D05", "Driver E", "PART_TIME", "13:00", "18:00", True,  False),
]

driver_df = pd.DataFrame(
    drivers,
    columns=[
        "driver_id",
        "driver_name",
        "employment",
        "shift_start",
        "shift_end",
        "can_small",
        "can_large",
    ]
)

driver_df["shift_start_min"] = (
    driver_df["shift_start"].map(clock_to_schedule_min)
)
driver_df["shift_end_min"] = (
    driver_df["shift_end"].map(clock_to_schedule_min)
)
driver_df["shift_hours"] = (
    driver_df["shift_end_min"]
    - driver_df["shift_start_min"]
) / 60.0

driver_df["qualification"] = np.where(
    driver_df["can_large"],
    "SMALL + LARGE",
    "SMALL only"
)

display(
    driver_df[[
        "driver_name",
        "employment",
        "qualification",
        "shift_start",
        "shift_end",
        "shift_hours",
    ]]
)


# ============================================================
# DRIVER x TRUCK COMPATIBILITY
# ============================================================

def can_drive(driver_row, truck_type):
    if truck_type == "LARGE":
        return bool(driver_row["can_large"])
    if truck_type == "SMALL":
        return bool(driver_row["can_small"])
    return False


compatibility_rows = []

for _, d in driver_df.iterrows():
    for _, t in truck_df.iterrows():
        compatibility_rows.append({
            "driver_id": d["driver_id"],
            "driver": d["driver_name"],
            "shift": f"{d['shift_start']}-{d['shift_end']}",
            "truck_id": int(t["vehicle_id"]),
            "truck": t["truck_name"],
            "truck_type": t["truck_type"],
            "qualified": can_drive(d, t["truck_type"]),
        })

driver_truck_compatibility_df = pd.DataFrame(
    compatibility_rows
)

print("Driver / Truck qualification matrix:")
display(
    driver_truck_compatibility_df.pivot(
        index=["driver", "shift"],
        columns=["truck", "truck_type"],
        values="qualified",
    )
)


# ============================================================
# PRE-DISPATCH STAFFING FILTER
# ============================================================
#
# The driver/truck roster is fixed BEFORE the VRP.
#
# We deliberately assign the earliest and latest full-time drivers
# to the two LARGE trucks so that LARGE-truck coverage spans most
# of the delivery day:
#
#   Truck 1 LARGE <- Driver A 06:00-15:00
#   Truck 2 LARGE <- Driver C 08:00-17:00
#   Truck 3 SMALL <- Driver B 07:00-16:00
#   Truck 4 SMALL <- Driver D 07:00-12:00
#   Truck 5 SMALL <- Driver E 13:00-18:00
#
# The qualification table above is still authoritative. If a planned
# pairing is not qualified, that physical truck is marked UNAVAILABLE
# and the VRP cannot assign customers to it.

planned_dispatch = pd.DataFrame([
    # vehicle_id, driver_id
    (0, "D01"),  # LARGE <- full-time / early
    (1, "D03"),  # LARGE <- full-time / late
    (2, "D02"),  # SMALL <- full-time / middle
    (3, "D04"),  # SMALL <- part-time / morning
    (4, "D05"),  # SMALL <- part-time / afternoon
], columns=[
    "vehicle_id",
    "driver_id",
])

assignment_rows = []

for _, plan in planned_dispatch.iterrows():
    vehicle_id = int(plan["vehicle_id"])
    driver_id = plan["driver_id"]

    truck = truck_df.loc[
        truck_df["vehicle_id"] == vehicle_id
    ].iloc[0]

    driver = driver_df.loc[
        driver_df["driver_id"] == driver_id
    ].iloc[0]

    qualified = can_drive(
        driver,
        truck["truck_type"],
    )

    assignment_rows.append({
        "vehicle_id": vehicle_id,
        "driver_id": driver_id if qualified else None,
        "available": bool(qualified),
        "staffing_note": (
            "QUALIFIED"
            if qualified
            else "UNAVAILABLE - DRIVER NOT QUALIFIED"
        ),
    })


truck_driver_assignment = pd.DataFrame(
    assignment_rows
)

vehicle_driver_df = (
    truck_df
    .merge(
        truck_driver_assignment,
        on="vehicle_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        driver_df,
        on="driver_id",
        how="left",
        validate="many_to_one",
    )
    .sort_values("vehicle_id")
    .reset_index(drop=True)
)

vehicle_driver_df["availability"] = np.where(
    vehicle_driver_df["available"].fillna(False),
    "AVAILABLE",
    "UNAVAILABLE",
)

# IMPORTANT:
# OR-Tools SetAllowedVehiclesForIndex() requires a Python sequence
# of native int values. Pandas/NumPy integer scalars can trigger:
#   TypeError: argument 2 of type 'absl::Span< int const >'
available_vehicle_ids = [
    int(v)
    for v in vehicle_driver_df.loc[
        vehicle_driver_df["available"].fillna(False),
        "vehicle_id",
    ].tolist()
]

print("Pre-dispatch truck availability after qualification filtering:")
display(
    vehicle_driver_df[[
        "vehicle_id",
        "truck_name",
        "truck_type",
        "availability",
        "driver_name",
        "employment",
        "qualification",
        "shift_start",
        "shift_end",
        "staffing_note",
    ]]
)

print(
    "Available trucks:",
    len(available_vehicle_ids),
    "/",
    len(truck_df),
)
print(
    "OR-Tools vehicle IDs:",
    available_vehicle_ids,
    "| Python types:",
    [type(v).__name__ for v in available_vehicle_ids],
)

if len(available_vehicle_ids) == 0:
    raise RuntimeError(
        "NO OPERATING TRUCKS: no qualified driver/truck pair is available."
    )


,driver_name,employment,qualification,shift_start,shift_end,shift_hours
0,Driver A,FULL_TIME,SMALL + LARGE,06:00,15:00,9.0
1,Driver B,FULL_TIME,SMALL + LARGE,07:00,16:00,9.0
2,Driver C,FULL_TIME,SMALL + LARGE,08:00,17:00,9.0
3,Driver D,PART_TIME,SMALL only,07:00,12:00,5.0
4,Driver E,PART_TIME,SMALL only,13:00,18:00,5.0


Driver / Truck qualification matrix:


,truck,Truck 1,Truck 2,Truck 3,Truck 4,Truck 5
,truck_type,LARGE,LARGE,SMALL,SMALL,SMALL
driver,shift,,,,,
Driver A,06:00-15:00,True,True,True,True,True
Driver B,07:00-16:00,True,True,True,True,True
Driver C,08:00-17:00,True,True,True,True,True
Driver D,07:00-12:00,False,False,True,True,True
Driver E,13:00-18:00,False,False,True,True,True


Pre-dispatch truck availability after qualification filtering:


,vehicle_id,truck_name,truck_type,availability,driver_name,employment,qualification,shift_start,shift_end,staffing_note
0,0,Truck 1,LARGE,AVAILABLE,Driver A,FULL_TIME,SMALL + LARGE,06:00,15:00,QUALIFIED
1,1,Truck 2,LARGE,AVAILABLE,Driver C,FULL_TIME,SMALL + LARGE,08:00,17:00,QUALIFIED
2,2,Truck 3,SMALL,AVAILABLE,Driver B,FULL_TIME,SMALL + LARGE,07:00,16:00,QUALIFIED
3,3,Truck 4,SMALL,AVAILABLE,Driver D,PART_TIME,SMALL only,07:00,12:00,QUALIFIED
4,4,Truck 5,SMALL,AVAILABLE,Driver E,PART_TIME,SMALL only,13:00,18:00,QUALIFIED


Available trucks: 5 / 5
OR-Tools vehicle IDs: [0, 1, 2, 3, 4] | Python types: ['int', 'int', 'int', 'int', 'int']


## 7. Convert order lines into VRP customer demand

For VRP, all order lines for the same customer are aggregated into:

- total shipment weight
- total aggregate volume
- total physical box count

This is the **fast routing-level capacity check**.

The real 3D geometry is checked later.

In [ ]:
customer_demand = (
    order_detail_df.groupby("customer_id", as_index=False)
    .agg(
        weight_kg=("line_weight_kg", "sum"),
        volume_liters=("line_volume_liters", "sum"),
        box_count=("quantity", "sum"),
        sku_count=("sku", "nunique")
    )
)

customer_df = customer_df.merge(
    customer_demand,
    on="customer_id",
    how="left"
)

customer_df[
    ["weight_kg", "volume_liters", "box_count", "sku_count"]
] = customer_df[
    ["weight_kg", "volume_liters", "box_count", "sku_count"]
].fillna(0)

customer_df["weight_kg"] = customer_df["weight_kg"].astype(int)
customer_df["volume_liters"] = customer_df["volume_liters"].round().astype(int)
customer_df["box_count"] = customer_df["box_count"].astype(int)
customer_df["sku_count"] = customer_df["sku_count"].astype(int)

display(customer_df[[
    "customer_id", "customer_name",
    "time_window", "service_min",
    "sku_count", "box_count",
    "weight_kg", "volume_liters"
]])

print("Total order weight:", int(customer_df["weight_kg"].sum()), "kg")
print(
    "Total aggregate order volume:",
    round(customer_df["volume_liters"].sum() / 1000, 2),
    "m³"
)

,customer_id,customer_name,time_window,service_min,sku_count,box_count,weight_kg,volume_liters
0,DEPOT,Depot / Warehouse,08:00-08:00,0,0,0,0,0
1,C01,Customer A,08:00-17:00,12,2,3,60,788
2,C02,Customer B,08:00-17:00,10,2,4,82,1056
3,C03,Customer C,12:00-16:00,12,2,3,106,1832
4,C04,Customer D,08:00-17:00,10,2,4,83,1472
5,C05,Customer E,09:00-12:00,10,2,3,74,1383
6,C06,Customer F,08:00-17:00,15,2,4,98,1538
7,C07,Customer G,11:00-14:00,10,2,3,96,1766
8,C08,Customer H,08:00-17:00,12,2,5,70,826
9,C09,Customer I,12:00-16:00,10,2,5,93,1459


Total order weight: 1721 kg
Total aggregate order volume: 28.23 m³


## 8. Expand ordered quantities into physical boxes

An order line such as:

```text
Customer B | SKU003 | quantity = 3
```

becomes three physical packing items:

```text
ORD-C02-SKU003-001
ORD-C02-SKU003-002
ORD-C02-SKU003-003
```

These individual boxes are what the 3D Packing algorithm actually loads into a truck.

In [ ]:
physical_items = []

for _, row in order_detail_df.iterrows():
    for unit_no in range(1, int(row["quantity"]) + 1):
        physical_items.append({
            "item_id": (
                f"{row['order_id']}-{row['sku']}-{unit_no:03d}"
            ),
            "order_id": row["order_id"],
            "customer_id": row["customer_id"],
            "customer_name": row["customer_name"],
            "sku": row["sku"],
            "product_name": row["product_name"],
            "shape": row["shape"],
            "length_cm": int(row["length_cm"]),
            "width_cm": int(row["width_cm"]),
            "height_cm": int(row["height_cm"]),
            "weight_kg": int(row["unit_weight_kg"]),
            "volume_liters": float(row["unit_volume_liters"])
        })

physical_item_df = pd.DataFrame(physical_items)

display(physical_item_df.head(20))

print("Physical boxes to deliver:", len(physical_item_df))
print(
    "Physical box weight total:",
    int(physical_item_df["weight_kg"].sum()),
    "kg"
)

,item_id,order_id,customer_id,customer_name,sku,product_name,shape,length_cm,width_cm,height_cm,weight_kg,volume_liters
0,ORD-C01-SKU001-001,ORD-C01,C01,Customer A,SKU001,Standard Carton S,BOX,60,40,35,12,84.0
1,ORD-C01-SKU001-002,ORD-C01,C01,Customer A,SKU001,Standard Carton S,BOX,60,40,35,12,84.0
2,ORD-C01-SKU002-001,ORD-C01,C01,Customer A,SKU002,Long Equipment A,BOX,310,50,40,36,620.0
3,ORD-C02-SKU003-001,ORD-C02,C02,Customer B,SKU003,Standard Carton M,BOX,80,55,45,18,198.0
4,ORD-C02-SKU003-002,ORD-C02,C02,Customer B,SKU003,Standard Carton M,BOX,80,55,45,18,198.0
5,ORD-C02-SKU003-003,ORD-C02,C02,Customer B,SKU003,Standard Carton M,BOX,80,55,45,18,198.0
6,ORD-C02-SKU004-001,ORD-C02,C02,Customer B,SKU004,Standard Carton L,BOX,110,70,60,28,462.0
7,ORD-C03-SKU005-001,ORD-C03,C03,Customer C,SKU005,Bulk Carton A,BOX,100,80,65,32,520.0
8,ORD-C03-SKU005-002,ORD-C03,C03,Customer C,SKU005,Bulk Carton A,BOX,100,80,65,32,520.0
9,ORD-C03-SKU006-001,ORD-C03,C03,Customer C,SKU006,Long Equipment B,BOX,320,55,45,42,792.0


Physical boxes to deliver: 76
Physical box weight total: 1721 kg


## 9. Plot the depot and customer locations

In [ ]:
center = [
    customer_df["lat"].mean(),
    customer_df["lon"].mean()
]

# IMPORTANT:
# Do not use tiles="OpenStreetMap"
# Do not use tiles="CartoDB positron"
point_map = folium.Map(
    location=center,
    zoom_start=11,
    tiles=None
)

# Same basemap that worked for the final route map
folium.TileLayer(
    tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap.DE",
    overlay=False,
    control=False,
    max_zoom=18
).add_to(point_map)


for _, row in customer_df.iterrows():

    if row.customer_id == "DEPOT":

        folium.Marker(
            [row.lat, row.lon],
            tooltip="DEPOT / WAREHOUSE",
            popup=(
                "<b>DEPOT / WAREHOUSE</b><br>"
                "Driver-dependent departure<br>"
                "Return within assigned driver shift"
            ),
            icon=folium.Icon(
                color="black",
                icon="home",
                prefix="fa"
            )
        ).add_to(point_map)

    else:

        customer_orders = order_detail_df[
            order_detail_df["customer_id"] == row.customer_id
        ]

        order_text = "<br>".join(
            f"{r.sku} {r.product_name} × {int(r.quantity)}"
            for _, r in customer_orders.iterrows()
        )

        popup = (
            f"<b>{row.customer_name} ({row.customer_id})</b><br>"
            f"{row.address}<br><br>"
            f"<b>Order</b><br>{order_text}<br><br>"
            f"Boxes: {row.box_count}<br>"
            f"Weight: {row.weight_kg} kg<br>"
            f"Aggregate volume: {row.volume_liters} L<br>"
            f"Time window: {row.time_window}<br>"
            f"Service: {row.service_min} min"
        )

        folium.CircleMarker(
            [row.lat, row.lon],
            radius=6,
            fill=True,
            fill_opacity=0.8,
            tooltip=(
                f"{row.customer_name} / "
                f"{row.customer_id}"
            ),
            popup=popup
        ).add_to(point_map)

point_map

# Chapter 2 — CHANGE: Replace TomTom with Public / Open Data

**Talk-script anchor: B1-OPEN-DATA**

**This is the main section to explain in Demo 3.**

The commercial TomTom layer is replaced by:

- OpenStreetMap / OSMnx
- City of Irvine official Speed + ADT GIS data
- Caltrans CWWP2 current highway travel times

The goal is to change the traffic-data layer while keeping the downstream optimization architecture essentially the same.

## 10. Open-data road model: Irvine GIS + Caltrans CWWP2 + OSM

The 07B1 road model is built in four layers:

1. **OSM base graph**
   - cached in Google Drive and reused
2. **City of Irvine Speed**
   - matched to OSM edges
   - used as the official local/arterial baseline speed where available
3. **City of Irvine ADT**
   - matched to OSM edges as metadata
   - not converted into fake real-time congestion
4. **Caltrans CWWP2 District 12 current travel time**
   - requested fresh each normal run
   - when a CWWP2 segment has usable endpoints inside the OSM graph, the observed travel time is compared with the baseline OSM/Irvine travel time
   - the resulting travel-time factor is applied to the corresponding OSM corridor

This creates:

```text
OSM network
    +
Irvine official street speed
    +
CWWP2 current highway travel time
    ↓
traffic_travel_time(edge)
    ↓
customer-to-customer matrix
    ↓
OR-Tools VRP
```

Sources:

- City of Irvine ADT/Speed REST service:  
  `https://gis.cityofirvine.org/arcgis/rest/services/ADT/MapServer`
- Caltrans CWWP2 District 12 current Travel Time JSON:  
  `https://cwwp2.dot.ca.gov/data/d12/tt/ttStatusD12.json`


In [ ]:
# ============================================================
# 10A. CONFIGURATION + GOOGLE DRIVE CACHE
# ============================================================

from google.colab import drive
from pathlib import Path
import gzip
import pickle
import shutil
import os
import json
import hashlib

drive.mount("/content/drive", force_remount=False)

# Google Drive cache directory.
# Change this path to a folder in your own Google Drive before running the notebook.
# Example:
#   /content/drive/MyDrive/delivery-vrp/tomtom_cache
CACHE_DIR_DRIVE = Path(
    "/content/drive/MyDrive/プレゼン発表/ExtremePython_2026-11-16/open_data_cache"
)

CACHE_DIR_LOCAL = Path(
    "/content/open_data_cache"
)

CACHE_DIR_DRIVE.mkdir(parents=True, exist_ok=True)
CACHE_DIR_LOCAL.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Data-source behavior
# ------------------------------------------------------------
USE_OSM = True
USE_IRVINE_GIS = True
USE_CWWP_CURRENT = True

# OSM road geometry changes slowly -> cache it.
FORCE_REFRESH_OSM = False

# Irvine Speed / ADT are not live feeds -> reuse by default.
REUSE_IRVINE_GIS_CACHE = True

# CWWP is current data -> fresh request every normal run.
REUSE_CWWP_CACHE = False

# If CWWP is temporarily unavailable, allow the latest saved
# response as a -safe fallback.
CWWP_FALLBACK_TO_LAST_CACHE = True

OSM_DOWNLOAD_RADIUS_M = 10_000

# CWWP travel-time factor sanity bounds.
CWWP_FACTOR_MIN = 0.70
CWWP_FACTOR_MAX = 4.00

IRVINE_BASE = (
    "https://gis.cityofirvine.org/"
    "arcgis/rest/services/ADT/MapServer"
)

IRVINE_SPEED_LAYER = 11
IRVINE_ADT_LAYER = 1

CWWP_D12_URL = (
    "https://cwwp2.dot.ca.gov/"
    "data/d12/tt/ttStatusD12.json"
)


def save_pickle_gz(path, obj):
    path = Path(path)
    tmp = Path(str(path) + ".tmp")

    with gzip.open(tmp, "wb") as f:
        pickle.dump(
            obj,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

    os.replace(tmp, path)


def load_pickle_gz(path):
    with gzip.open(path, "rb") as f:
        return pickle.load(f)


def copy_drive_to_local_if_needed(drive_path, local_path):
    drive_path = Path(drive_path)
    local_path = Path(local_path)

    if local_path.exists():
        return "LOCAL"

    if drive_path.exists():
        shutil.copy2(
            drive_path,
            local_path
        )
        return "GOOGLE DRIVE"

    return None


# ------------------------------------------------------------
# Cache names
# ------------------------------------------------------------
depot_lat = float(customer_df.loc[0, "lat"])
depot_lon = float(customer_df.loc[0, "lon"])

osm_signature_payload = {
    "depot_lat": round(depot_lat, 6),
    "depot_lon": round(depot_lon, 6),
    "radius_m": OSM_DOWNLOAD_RADIUS_M,
    "network_type": "drive",
}

osm_signature = hashlib.sha256(
    json.dumps(
        osm_signature_payload,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()[:12]

OSM_CACHE_NAME = (
    f"06B1_osm_base_{osm_signature}.pkl.gz"
)

OSM_CACHE_FILE_LOCAL = (
    CACHE_DIR_LOCAL / OSM_CACHE_NAME
)

OSM_CACHE_FILE_DRIVE = (
    CACHE_DIR_DRIVE / OSM_CACHE_NAME
)

IRVINE_CACHE_FILE_LOCAL = (
    CACHE_DIR_LOCAL / "06B1_irvine_speed_adt.pkl.gz"
)

IRVINE_CACHE_FILE_DRIVE = (
    CACHE_DIR_DRIVE / "06B1_irvine_speed_adt.pkl.gz"
)

CWWP_CACHE_FILE_LOCAL = (
    CACHE_DIR_LOCAL / "06B1_cwwp_d12_latest.pkl.gz"
)

CWWP_CACHE_FILE_DRIVE = (
    CACHE_DIR_DRIVE / "06B1_cwwp_d12_latest.pkl.gz"
)

MERGED_SNAPSHOT_FILE_DRIVE = (
    CACHE_DIR_DRIVE / "06B1_latest_merged_snapshot.pkl.gz"
)

print("Open-data cache directory:")
print(CACHE_DIR_DRIVE)


ValueError: mount failed

In [ ]:
# ============================================================
# 10B. CITY OF IRVINE ARCGIS REST
#      Speed layer + 2022 ADT
# ============================================================

def fetch_arcgis_geojson_all(
    layer_id,
    out_fields="*",
    batch_size=2000,
    timeout=60
):
    """
    Download all features from an ArcGIS layer as one GeoJSON
    FeatureCollection using REST pagination.
    """
    url = (
        f"{IRVINE_BASE}/{layer_id}/query"
    )

    features = []
    offset = 0
    crs = None

    while True:
        params = {
            "where": "1=1",
            "outFields": out_fields,
            "returnGeometry": "true",
            "outSR": "4326",
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": batch_size,
            "orderByFields": "OBJECTID",
        }

        r = requests.get(
            url,
            params=params,
            timeout=timeout
        )
        r.raise_for_status()

        page = r.json()

        page_features = page.get(
            "features",
            []
        )

        if crs is None:
            crs = page.get("crs")

        features.extend(
            page_features
        )

        if len(page_features) < batch_size:
            break

        offset += batch_size

    return {
        "type": "FeatureCollection",
        "features": features,
        "crs": crs,
    }


irvine_bundle = None
irvine_cache_source = None

if REUSE_IRVINE_GIS_CACHE:
    irvine_cache_source = (
        copy_drive_to_local_if_needed(
            IRVINE_CACHE_FILE_DRIVE,
            IRVINE_CACHE_FILE_LOCAL
        )
    )

    if irvine_cache_source:
        try:
            irvine_bundle = load_pickle_gz(
                IRVINE_CACHE_FILE_LOCAL
            )
            print(
                f"Irvine GIS loaded from {irvine_cache_source}"
            )
        except Exception as exc:
            print(
                "Irvine GIS cache load failed:",
                repr(exc)
            )
            irvine_bundle = None


if irvine_bundle is None:

    if not USE_IRVINE_GIS:
        irvine_speed_geojson = {
            "type": "FeatureCollection",
            "features": [],
        }
        irvine_adt_geojson = {
            "type": "FeatureCollection",
            "features": [],
        }

    else:
        print(
            "Downloading City of Irvine Speed layer..."
        )

        irvine_speed_geojson = (
            fetch_arcgis_geojson_all(
                IRVINE_SPEED_LAYER,
                out_fields=(
                    "OBJECTID,"
                    "STREETNAME,"
                    "Speed,"
                    "CL"
                )
            )
        )

        print(
            "Downloading City of Irvine 2022 ADT layer..."
        )

        irvine_adt_geojson = (
            fetch_arcgis_geojson_all(
                IRVINE_ADT_LAYER,
                out_fields=(
                    "OBJECTID,"
                    "STREETNAME,"
                    "Y2022"
                )
            )
        )

    irvine_bundle = {
        "downloaded_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
        "speed":
            irvine_speed_geojson,
        "adt":
            irvine_adt_geojson,
    }

    save_pickle_gz(
        IRVINE_CACHE_FILE_LOCAL,
        irvine_bundle
    )

    shutil.copy2(
        IRVINE_CACHE_FILE_LOCAL,
        IRVINE_CACHE_FILE_DRIVE
    )

    print(
        "Irvine GIS saved to Google Drive."
    )

else:
    irvine_speed_geojson = (
        irvine_bundle["speed"]
    )
    irvine_adt_geojson = (
        irvine_bundle["adt"]
    )


# Ensure these variables exist in both code paths.
irvine_speed_geojson = (
    irvine_bundle["speed"]
)
irvine_adt_geojson = (
    irvine_bundle["adt"]
)

print()
print(
    "Irvine Speed features:",
    len(
        irvine_speed_geojson.get(
            "features",
            []
        )
    )
)

print(
    "Irvine ADT features:",
    len(
        irvine_adt_geojson.get(
            "features",
            []
        )
    )
)


In [ ]:
# ============================================================
# 10C. CALTRANS CWWP2 DISTRICT 12
#      CURRENT travel-time data
# ============================================================

def _num_from_dict(d, candidate_names):
    """
    Case-insensitive numeric extraction from a dict.
    Used because CWWP field naming may differ by feed version.
    """
    if not isinstance(d, dict):
        return np.nan

    lookup = {
        str(k).lower(): v
        for k, v in d.items()
    }

    for name in candidate_names:
        key = name.lower()

        if key in lookup:
            try:
                return float(
                    lookup[key]
                )
            except Exception:
                pass

    return np.nan


def _text_from_dict(d, candidate_names):
    if not isinstance(d, dict):
        return None

    lookup = {
        str(k).lower(): v
        for k, v in d.items()
    }

    for name in candidate_names:
        key = name.lower()

        if key in lookup:
            value = lookup[key]

            if value is not None:
                return str(value)

    return None


def parse_cwwp_tt(payload):
    """
    Parse the District 12 travel-time JSON into a tolerant DataFrame.
    Known CWWP structure:
        data[] -> tt -> location / traveltime
    """
    if isinstance(payload, dict):
        items = payload.get(
            "data",
            []
        )
    elif isinstance(payload, list):
        items = payload
    else:
        items = []

    rows = []

    for item in items:
        if not isinstance(item, dict):
            continue

        tt = item.get(
            "tt",
            item
        )

        if not isinstance(tt, dict):
            continue

        location = tt.get(
            "location",
            {}
        )

        travel = tt.get(
            "traveltime",
            tt.get(
                "travelTime",
                {}
            )
        )

        begin = (
            location.get(
                "begin",
                {}
            )
            if isinstance(
                location,
                dict
            )
            else {}
        )

        end = (
            location.get(
                "end",
                {}
            )
            if isinstance(
                location,
                dict
            )
            else {}
        )

        route = (
            _text_from_dict(
                begin,
                [
                    "beginRoute",
                    "route"
                ]
            )
            or _text_from_dict(
                location,
                ["route"]
            )
        )

        begin_name = _text_from_dict(
            begin,
            [
                "beginLocationName",
                "locationName",
                "name"
            ]
        )

        end_name = _text_from_dict(
            end,
            [
                "endLocationName",
                "locationName",
                "name"
            ]
        )

        direction = _text_from_dict(
            location,
            [
                "travelFlowDirection",
                "direction"
            ]
        )

        observed_sec = _num_from_dict(
            travel,
            [
                "calculatedTraveltime",
                "calculatedTravelTime",
                "travelTime",
                "traveltime"
            ]
        )

        begin_lat = _num_from_dict(
            begin,
            [
                "beginLatitude",
                "latitude",
                "lat"
            ]
        )

        begin_lon = _num_from_dict(
            begin,
            [
                "beginLongitude",
                "longitude",
                "lon",
                "lng"
            ]
        )

        end_lat = _num_from_dict(
            end,
            [
                "endLatitude",
                "latitude",
                "lat"
            ]
        )

        end_lon = _num_from_dict(
            end,
            [
                "endLongitude",
                "longitude",
                "lon",
                "lng"
            ]
        )

        begin_postmile = _num_from_dict(
            begin,
            [
                "beginPostmile",
                "postmile"
            ]
        )

        end_postmile = _num_from_dict(
            end,
            [
                "endPostmile",
                "postmile"
            ]
        )

        timestamp = None

        if isinstance(
            travel,
            dict
        ):
            ts = travel.get(
                "traveltimeTimestamp",
                travel.get(
                    "travelTimeTimestamp"
                )
            )

            if isinstance(ts, dict):
                timestamp = (
                    str(
                        ts.get(
                            "traveltimeDate",
                            ts.get(
                                "date",
                                ""
                            )
                        )
                    )
                    + " "
                    + str(
                        ts.get(
                            "traveltimeTime",
                            ts.get(
                                "time",
                                ""
                            )
                        )
                    )
                ).strip()

        rows.append({
            "route":
                route,
            "direction":
                direction,
            "begin_name":
                begin_name,
            "end_name":
                end_name,
            "begin_postmile":
                begin_postmile,
            "end_postmile":
                end_postmile,
            "begin_lat":
                begin_lat,
            "begin_lon":
                begin_lon,
            "end_lat":
                end_lat,
            "end_lon":
                end_lon,
            "current_travel_time_sec":
                observed_sec,
            "timestamp":
                timestamp,
        })

    return pd.DataFrame(rows)


cwwp_payload = None
cwwp_source = None
cwwp_download_utc = None

if REUSE_CWWP_CACHE:

    source = copy_drive_to_local_if_needed(
        CWWP_CACHE_FILE_DRIVE,
        CWWP_CACHE_FILE_LOCAL
    )

    if source:
        try:
            bundle = load_pickle_gz(
                CWWP_CACHE_FILE_LOCAL
            )
            cwwp_payload = bundle[
                "payload"
            ]
            cwwp_source = (
                f"CACHE: {source}"
            )
            cwwp_download_utc = bundle.get(
                "downloaded_utc"
            )
        except Exception:
            cwwp_payload = None


if cwwp_payload is None and USE_CWWP_CURRENT:

    try:
        print(
            "Downloading current Caltrans CWWP2 District 12 travel times..."
        )

        r = requests.get(
            CWWP_D12_URL,
            timeout=45,
            headers={
                "User-Agent":
                    "Mozilla/5.0 06B1-XtremePython-demo"
            }
        )

        r.raise_for_status()

        cwwp_payload = r.json()
        cwwp_download_utc = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )
        cwwp_source = "LIVE CWWP2"

        bundle = {
            "downloaded_utc":
                cwwp_download_utc,
            "payload":
                cwwp_payload,
        }

        save_pickle_gz(
            CWWP_CACHE_FILE_LOCAL,
            bundle
        )

        shutil.copy2(
            CWWP_CACHE_FILE_LOCAL,
            CWWP_CACHE_FILE_DRIVE
        )

    except Exception as exc:

        print(
            "Live CWWP2 request failed:",
            repr(exc)
        )

        if (
            CWWP_FALLBACK_TO_LAST_CACHE
            and CWWP_CACHE_FILE_DRIVE.exists()
        ):
            try:
                shutil.copy2(
                    CWWP_CACHE_FILE_DRIVE,
                    CWWP_CACHE_FILE_LOCAL
                )

                bundle = load_pickle_gz(
                    CWWP_CACHE_FILE_LOCAL
                )

                cwwp_payload = bundle[
                    "payload"
                ]

                cwwp_download_utc = bundle.get(
                    "downloaded_utc"
                )

                cwwp_source = (
                    "FALLBACK SAVED CWWP2"
                )

                print(
                    "Using last saved CWWP2 response."
                )

            except Exception as cache_exc:
                print(
                    "Saved CWWP2 fallback also failed:",
                    repr(cache_exc)
                )


if cwwp_payload is None:
    cwwp_payload = {
        "data": []
    }

cwwp_df = parse_cwwp_tt(
    cwwp_payload
)

if len(cwwp_df):
    valid_coord_mask = (
        cwwp_df[
            [
                "begin_lat",
                "begin_lon",
                "end_lat",
                "end_lon"
            ]
        ]
        .notna()
        .all(axis=1)
    )
else:
    valid_coord_mask = pd.Series(
        dtype=bool
    )

print()
print(
    "CWWP source:",
    cwwp_source
)

print(
    "CWWP downloaded UTC:",
    cwwp_download_utc
)

print(
    "Travel-time records:",
    len(cwwp_df)
)

print(
    "Records with endpoint coordinates:",
    int(
        valid_coord_mask.sum()
    )
)

display(
    cwwp_df.head(15)
)


In [ ]:
# ============================================================
# 10D. BASE OSM GRAPH — CACHED
# ============================================================

straight_km = build_haversine_matrix(
    customer_df
)

G = None
osm_nodes = None

osm_cache_source = None

if not FORCE_REFRESH_OSM:

    osm_cache_source = (
        copy_drive_to_local_if_needed(
            OSM_CACHE_FILE_DRIVE,
            OSM_CACHE_FILE_LOCAL
        )
    )

    if osm_cache_source:
        try:
            osm_bundle = load_pickle_gz(
                OSM_CACHE_FILE_LOCAL
            )

            if (
                osm_bundle.get(
                    "osm_signature"
                )
                == osm_signature
            ):
                G = osm_bundle[
                    "G"
                ]

                print(
                    f"Loaded OSM graph from {osm_cache_source}"
                )

        except Exception as exc:
            print(
                "OSM cache load failed:",
                repr(exc)
            )
            G = None


if FORCE_REFRESH_OSM or G is None:

    if not USE_OSM:
        raise RuntimeError(
            "OSM is disabled and no cached graph exists."
        )

    print(
        "Downloading/building OSM road graph..."
    )

    G = ox.graph.graph_from_point(
        (
            depot_lat,
            depot_lon
        ),
        dist=OSM_DOWNLOAD_RADIUS_M,
        network_type="drive",
        simplify=True
    )

    G = ox.routing.add_edge_speeds(
        G
    )

    G = ox.routing.add_edge_travel_times(
        G
    )

    osm_bundle = {
        "version":
            "06B1-open-data-v1",
        "osm_signature":
            osm_signature,
        "G":
            G,
    }

    save_pickle_gz(
        OSM_CACHE_FILE_LOCAL,
        osm_bundle
    )

    shutil.copy2(
        OSM_CACHE_FILE_LOCAL,
        OSM_CACHE_FILE_DRIVE
    )

    print(
        "Base OSM graph saved to Google Drive."
    )


# ------------------------------------------------------------
# Make the directed road graph mutually reachable.
#
# OSM drive graphs can contain tiny disconnected / one-way-only
# components. If a customer snaps to one of those components,
# Dijkstra may not contain another customer node and a KeyError
# occurs later.
#
# Keep the largest strongly connected component, then snap all
# customers to that routable component.
# ------------------------------------------------------------
original_node_count = len(G)
original_edge_count = len(G.edges)

if not nx.is_strongly_connected(G):
    largest_scc_nodes = max(
        nx.strongly_connected_components(G),
        key=len
    )

    G = G.subgraph(
        largest_scc_nodes
    ).copy()

    print()
    print(
        "OSM graph connectivity cleanup:"
    )
    print(
        "  Before:",
        f"{original_node_count:,} nodes / "
        f"{original_edge_count:,} edges"
    )
    print(
        "  After :",
        f"{len(G):,} nodes / "
        f"{len(G.edges):,} edges"
    )

osm_nodes = ox.distance.nearest_nodes(
    G,
    X=customer_df[
        "lon"
    ].to_numpy(),
    Y=customer_df[
        "lat"
    ].to_numpy()
)

print()
print(
    "OSM graph ready:",
    f"{len(G):,} nodes, "
    f"{len(G.edges):,} directed edges"
)
print(
    "Strongly connected:",
    nx.is_strongly_connected(G)
)


In [ ]:
# ============================================================
# 10E. APPLY IRVINE SPEED / ADT + CWWP CURRENT TRAVEL TIME
#      THEN BUILD CUSTOMER MATRICES
# ============================================================

def geojson_lines(feature):
    """
    Return line coordinate arrays as [(lat, lon), ...].
    ArcGIS GeoJSON coordinates arrive as [lon, lat].
    """
    geom = feature.get(
        "geometry",
        {}
    )

    gtype = geom.get(
        "type"
    )

    coords = geom.get(
        "coordinates",
        []
    )

    if gtype == "LineString":
        raw_lines = [coords]

    elif gtype == "MultiLineString":
        raw_lines = coords

    else:
        return []

    result = []

    for raw in raw_lines:
        line = []

        for xy in raw:
            if (
                isinstance(xy, (list, tuple))
                and len(xy) >= 2
            ):
                lon = float(xy[0])
                lat = float(xy[1])

                line.append(
                    (lat, lon)
                )

        if len(line) >= 2:
            result.append(
                line
            )

    return result


def sample_line_points(
    line,
    max_points=10
):
    if len(line) < 2:
        return line

    pts = list(line)

    pts.extend([
        (
            (a[0] + b[0]) / 2.0,
            (a[1] + b[1]) / 2.0,
        )
        for a, b in zip(
            line[:-1],
            line[1:]
        )
    ])

    if len(pts) <= max_points:
        return pts

    idx = np.linspace(
        0,
        len(pts) - 1,
        max_points
    ).round().astype(int)

    return [
        pts[i]
        for i in idx
    ]


def batch_match_geojson_to_edges(
    feature_collection,
    value_field,
    extra_fields=(),
    max_points=8
):
    """
    Approximate GeoJSON line -> OSM edge matching.

    Returns:
      edge_values[(u,v,k)] = list of numeric values
      display_segments     = line features for Folium diagnostics
    """
    sample_edges_feature = []
    sample_lats = []
    sample_lons = []

    feature_values = []
    display_segments = []

    for feature_id, feature in enumerate(
        feature_collection.get(
            "features",
            []
        )
    ):
        props = feature.get(
            "properties",
            {}
        )

        try:
            value = float(
                props.get(
                    value_field
                )
            )
        except Exception:
            continue

        if not np.isfinite(value):
            continue

        lines = geojson_lines(
            feature
        )

        feature_values.append(
            (
                feature_id,
                value,
                props,
                lines
            )
        )

        for line in lines:

            display_segments.append({
                "coords":
                    line,
                "value":
                    value,
                "properties":
                    {
                        k:
                            props.get(k)
                        for k in extra_fields
                    },
            })

            for lat, lon in sample_line_points(
                line,
                max_points=max_points
            ):
                sample_edges_feature.append(
                    (
                        feature_id,
                        value,
                        props
                    )
                )
                sample_lats.append(
                    lat
                )
                sample_lons.append(
                    lon
                )

    edge_values = defaultdict(
        list
    )

    edge_props = defaultdict(
        list
    )

    if sample_lats:

        nearest = ox.distance.nearest_edges(
            G,
            X=np.asarray(
                sample_lons,
                dtype=float
            ),
            Y=np.asarray(
                sample_lats,
                dtype=float
            )
        )

        if (
            isinstance(nearest, tuple)
            and len(nearest) == 3
            and np.isscalar(nearest[0])
        ):
            nearest = [nearest]

        for (
            item,
            edge
        ) in zip(
            sample_edges_feature,
            nearest
        ):
            feature_id, value, props = item
            edge = tuple(edge)

            if len(edge) != 3:
                continue

            edge_values[
                edge
            ].append(
                value
            )

            edge_props[
                edge
            ].append(
                props
            )

    return (
        edge_values,
        edge_props,
        display_segments
    )


# ------------------------------------------------------------
# Initialize base times
# ------------------------------------------------------------
for (
    u,
    v,
    k,
    data
) in G.edges(
    keys=True,
    data=True
):
    osm_time = float(
        data.get(
            "travel_time",
            0.0
        )
    )

    data[
        "osm_travel_time"
    ] = osm_time

    data[
        "open_base_travel_time"
    ] = osm_time

    data[
        "traffic_travel_time"
    ] = osm_time

    data[
        "irvine_speed_mph"
    ] = np.nan

    data[
        "irvine_adt_2022"
    ] = np.nan

    data[
        "cwwp_factor"
    ] = 1.0

    data[
        "cwwp_matched"
    ] = False


# ------------------------------------------------------------
# City of Irvine official Speed -> OSM
# ------------------------------------------------------------
(
    speed_edge_values,
    speed_edge_props,
    irvine_speed_segments
) = batch_match_geojson_to_edges(
    irvine_speed_geojson,
    value_field="Speed",
    extra_fields=(
        "STREETNAME",
        "CL"
    ),
    max_points=8
)

speed_matched_edges = 0

for edge, values in speed_edge_values.items():

    u, v, k = edge

    if not G.has_edge(
        u,
        v,
        k
    ):
        continue

    speed_mph = float(
        np.median(values)
    )

    if (
        not np.isfinite(speed_mph)
        or speed_mph <= 0
    ):
        continue

    data = G.edges[
        u,
        v,
        k
    ]

    length_m = float(
        data.get(
            "length",
            0.0
        )
    )

    data[
        "irvine_speed_mph"
    ] = speed_mph

    data[
        "open_base_travel_time"
    ] = (
        length_m
        / (
            speed_mph
            * 1609.344
            / 3600.0
        )
    )

    data[
        "traffic_travel_time"
    ] = data[
        "open_base_travel_time"
    ]

    speed_matched_edges += 1


# ------------------------------------------------------------
# City of Irvine ADT -> OSM metadata
# NOTE: ADT is NOT converted into a real-time congestion factor.
# ------------------------------------------------------------
(
    adt_edge_values,
    adt_edge_props,
    irvine_adt_segments
) = batch_match_geojson_to_edges(
    irvine_adt_geojson,
    value_field="Y2022",
    extra_fields=(
        "STREETNAME",
    ),
    max_points=6
)

adt_matched_edges = 0

for edge, values in adt_edge_values.items():

    u, v, k = edge

    if not G.has_edge(
        u,
        v,
        k
    ):
        continue

    adt = float(
        np.median(values)
    )

    G.edges[
        u,
        v,
        k
    ][
        "irvine_adt_2022"
    ] = adt

    adt_matched_edges += 1


# ------------------------------------------------------------
# CWWP current Travel Time -> OSM corridor factor
# ------------------------------------------------------------
# First get graph geographic bounds so distant District 12
# records do not get snapped to the edge of our Irvine graph.
graph_lats = np.asarray([
    data["y"]
    for _, data in G.nodes(
        data=True
    )
])

graph_lons = np.asarray([
    data["x"]
    for _, data in G.nodes(
        data=True
    )
])

graph_south = float(
    graph_lats.min()
)
graph_north = float(
    graph_lats.max()
)
graph_west = float(
    graph_lons.min()
)
graph_east = float(
    graph_lons.max()
)

bbox_pad = 0.015

cwwp_edge_factors = defaultdict(
    list
)

cwwp_corridors = []


def edge_key_for_step(
    graph,
    u,
    v,
    weight_name
):
    """
    Choose the parallel edge with the smallest requested weight.
    """
    candidates = graph.get_edge_data(
        u,
        v,
        default={}
    )

    if not candidates:
        return None

    best_k = min(
        candidates.keys(),
        key=lambda kk: float(
            candidates[kk].get(
                weight_name,
                1e18
            )
        )
    )

    return (
        u,
        v,
        best_k
    )


cwwp_records_considered = 0
cwwp_records_applied = 0

for _, row in cwwp_df.iterrows():

    values = [
        row.get("begin_lat"),
        row.get("begin_lon"),
        row.get("end_lat"),
        row.get("end_lon"),
        row.get("current_travel_time_sec"),
    ]

    try:
        (
            begin_lat,
            begin_lon,
            end_lat,
            end_lon,
            observed_sec,
        ) = [
            float(x)
            for x in values
        ]
    except Exception:
        continue

    if not all(
        np.isfinite(
            [
                begin_lat,
                begin_lon,
                end_lat,
                end_lon,
                observed_sec,
            ]
        )
    ):
        continue

    if observed_sec <= 0:
        continue

    # Reject placeholder/invalid coordinates.
    if not (
        -90 <= begin_lat <= 90
        and -180 <= begin_lon <= 180
        and -90 <= end_lat <= 90
        and -180 <= end_lon <= 180
    ):
        continue

    # Only use CWWP records physically inside / very near
    # the OSM graph downloaded for this demo.
    if not (
        graph_south - bbox_pad
        <= begin_lat
        <= graph_north + bbox_pad
        and graph_west - bbox_pad
        <= begin_lon
        <= graph_east + bbox_pad
        and graph_south - bbox_pad
        <= end_lat
        <= graph_north + bbox_pad
        and graph_west - bbox_pad
        <= end_lon
        <= graph_east + bbox_pad
    ):
        continue

    cwwp_records_considered += 1

    try:
        begin_node = ox.distance.nearest_nodes(
            G,
            X=begin_lon,
            Y=begin_lat
        )

        end_node = ox.distance.nearest_nodes(
            G,
            X=end_lon,
            Y=end_lat
        )

        path = nx.shortest_path(
            G,
            begin_node,
            end_node,
            weight="open_base_travel_time"
        )

        if len(path) < 2:
            continue

        path_edges = []

        baseline_sec = 0.0

        for a, b in zip(
            path[:-1],
            path[1:]
        ):
            edge = edge_key_for_step(
                G,
                a,
                b,
                "open_base_travel_time"
            )

            if edge is None:
                continue

            data = G.edges[
                edge
            ]

            baseline_sec += float(
                data.get(
                    "open_base_travel_time",
                    0.0
                )
            )

            path_edges.append(
                edge
            )

        if (
            baseline_sec <= 0
            or not path_edges
        ):
            continue

        raw_factor = (
            observed_sec
            / baseline_sec
        )

        factor = float(
            np.clip(
                raw_factor,
                CWWP_FACTOR_MIN,
                CWWP_FACTOR_MAX
            )
        )

        for edge in path_edges:
            cwwp_edge_factors[
                edge
            ].append(
                factor
            )

        coords = [
            (
                G.nodes[node]["y"],
                G.nodes[node]["x"]
            )
            for node in path
        ]

        cwwp_corridors.append({
            "coords":
                coords,
            "route":
                row.get("route"),
            "direction":
                row.get("direction"),
            "begin_name":
                row.get("begin_name"),
            "end_name":
                row.get("end_name"),
            "observed_sec":
                observed_sec,
            "baseline_sec":
                baseline_sec,
            "raw_factor":
                raw_factor,
            "factor":
                factor,
            "timestamp":
                row.get("timestamp"),
        })

        cwwp_records_applied += 1

    except Exception:
        continue


for edge, factors in cwwp_edge_factors.items():

    u, v, k = edge

    if not G.has_edge(
        u,
        v,
        k
    ):
        continue

    factor = float(
        np.median(
            factors
        )
    )

    data = G.edges[
        u,
        v,
        k
    ]

    data[
        "cwwp_factor"
    ] = factor

    data[
        "cwwp_matched"
    ] = True

    data[
        "traffic_travel_time"
    ] = (
        float(
            data.get(
                "open_base_travel_time",
                data.get(
                    "travel_time",
                    0.0
                )
            )
        )
        * factor
    )


# ------------------------------------------------------------
# Customer-to-customer matrices
# ------------------------------------------------------------
n = len(
    customer_df
)

baseline_road_time_min = np.zeros(
    (n, n),
    dtype=int
)

traffic_road_time_min = np.zeros(
    (n, n),
    dtype=int
)

road_distance_km = np.zeros(
    (n, n),
    dtype=float
)

unreachable_pair_fallback_count = 0

for (
    i,
    source_node
) in enumerate(
    osm_nodes
):

    base_times = (
        nx.single_source_dijkstra_path_length(
            G,
            source_node,
            weight="open_base_travel_time"
        )
    )

    traffic_times = (
        nx.single_source_dijkstra_path_length(
            G,
            source_node,
            weight="traffic_travel_time"
        )
    )

    distances = (
        nx.single_source_dijkstra_path_length(
            G,
            source_node,
            weight="length"
        )
    )

    for (
        j,
        target_node
    ) in enumerate(
        osm_nodes
    ):

        # A strongly connected graph should make all three values
        # available. Use .get() anyway so the notebook remains
        # -safe if OSM data contains an unexpected gap.
        base_sec = base_times.get(
            target_node
        )

        traffic_sec = traffic_times.get(
            target_node
        )

        distance_m = distances.get(
            target_node
        )

        if (
            base_sec is None
            or traffic_sec is None
            or distance_m is None
        ):
            # Safe fallback for an unreachable OSM pair.
            #
            # Use the same simple model used elsewhere in the
            # notebook: straight-line × 1.25 at 35 km/h.
            unreachable_pair_fallback_count += 1

            fallback_km = float(
                straight_km[
                    i,
                    j
                ]
                * 1.25
            )

            fallback_min = int(
                round(
                    (
                        fallback_km
                        / 35.0
                    )
                    * 60.0
                )
            )

            baseline_road_time_min[
                i,
                j
            ] = max(
                0,
                fallback_min
            )

            traffic_road_time_min[
                i,
                j
            ] = max(
                0,
                fallback_min
            )

            road_distance_km[
                i,
                j
            ] = fallback_km

            print(
                "WARNING: OSM pair unreachable; "
                "fallback used:",
                customer_df.loc[
                    i,
                    "customer_id"
                ],
                "->",
                customer_df.loc[
                    j,
                    "customer_id"
                ]
            )

        else:
            baseline_road_time_min[
                i,
                j
            ] = max(
                0,
                int(
                    round(
                        float(
                            base_sec
                        )
                        / 60.0
                    )
                )
            )

            traffic_road_time_min[
                i,
                j
            ] = max(
                0,
                int(
                    round(
                        float(
                            traffic_sec
                        )
                        / 60.0
                    )
                )
            )

            road_distance_km[
                i,
                j
            ] = (
                float(
                    distance_m
                )
                / 1000.0
            )


road_time_min = (
    traffic_road_time_min.copy()
)

road_mode = (
    "OSMnx + Irvine official Speed "
    "+ Caltrans CWWP2 current highway travel time"
)

mask = ~np.eye(
    n,
    dtype=bool
)

delta = (
    traffic_road_time_min[
        mask
    ]
    - baseline_road_time_min[
        mask
    ]
)

print("=" * 72)
print("OPEN-DATA ROAD MODEL READY")
print("=" * 72)

print(
    "Irvine Speed matched edges:",
    speed_matched_edges
)

print(
    "Irvine ADT matched edges:",
    adt_matched_edges
)

print(
    "CWWP records inside graph:",
    cwwp_records_considered
)

print(
    "CWWP corridors applied:",
    cwwp_records_applied
)

print(
    "OSM edges with CWWP factor:",
    len(
        cwwp_edge_factors
    )
)

print(
    "Road model:",
    road_mode
)

print(
    "Unreachable OSM pairs using fallback:",
    unreachable_pair_fallback_count
)

print(
    "Pairs with changed travel time:",
    int(
        np.count_nonzero(
            delta
        )
    )
)

print(
    "Mean current-traffic impact (min):",
    round(
        float(
            delta.mean()
        ),
        2
    )
)

print(
    "Max current delay (min):",
    int(
        delta.max()
    )
)

print(
    "Max current saving (min):",
    int(
        delta.min()
    )
)

# Save latest merged state ONLY as a replay/debug snapshot.
# It is NOT automatically reused as live traffic.
merged_snapshot = {
    "saved_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "cwwp_source":
        cwwp_source,
    "cwwp_download_utc":
        cwwp_download_utc,
    "G":
        G,
    "osm_nodes":
        osm_nodes,
    "baseline_road_time_min":
        baseline_road_time_min,
    "traffic_road_time_min":
        traffic_road_time_min,
    "road_distance_km":
        road_distance_km,
    "road_time_min":
        road_time_min,
    "road_mode":
        road_mode,
    "cwwp_corridors":
        cwwp_corridors,
}

save_pickle_gz(
    MERGED_SNAPSHOT_FILE_DRIVE,
    merged_snapshot
)

if (
    USE_CWWP_CURRENT
    and cwwp_records_applied == 0
):
    print()
    print(
        "NOTE: CWWP data were downloaded, but no current "
        "travel-time corridor could be geolocated inside this "
        "OSM graph from the available endpoint fields."
    )
    print(
        "The VRP is still valid, using OSM + Irvine official "
        "Speed; ADT remains metadata only."
    )


In [ ]:
# ============================================================
# 10F. OPEN-DATA DIAGNOSTICS
# ============================================================

print("=" * 72)
print("06B1 OPEN DATA DIAGNOSTICS")
print("=" * 72)

print()
print("City of Irvine GIS:")
print(
    "  Speed features:",
    len(
        irvine_speed_geojson.get(
            "features",
            []
        )
    )
)
print(
    "  ADT features:",
    len(
        irvine_adt_geojson.get(
            "features",
            []
        )
    )
)

print()
print("Caltrans CWWP2:")
print(
    "  Source:",
    cwwp_source
)
print(
    "  Download UTC:",
    cwwp_download_utc
)
print(
    "  Records:",
    len(cwwp_df)
)
print(
    "  Applied corridors:",
    cwwp_records_applied
)

print()
print("OSM cache:")
print(
    "  LOCAL:",
    OSM_CACHE_FILE_LOCAL.exists()
)
print(
    "  DRIVE:",
    OSM_CACHE_FILE_DRIVE.exists()
)

print()
print("Irvine GIS cache:")
print(
    "  LOCAL:",
    IRVINE_CACHE_FILE_LOCAL.exists()
)
print(
    "  DRIVE:",
    IRVINE_CACHE_FILE_DRIVE.exists()
)

print()
print(
    "Latest merged replay snapshot:",
    MERGED_SNAPSHOT_FILE_DRIVE
)


### 10.1 Visualize the open-data road model

This diagnostic map separates the data sources:

- **City of Irvine official Speed** — local/arterial baseline speed
- **Caltrans CWWP2** — current highway travel-time corridors that could be mapped into this OSM graph
- **Customers / depot**

CWWP corridor colors are based on the current-travel-time factor compared with the OSM/Irvine baseline:

- green: `<= 1.05`
- orange: `1.05–1.25`
- dark orange: `1.25–1.50`
- red: `> 1.50`

ADT is retained on OSM edges as `irvine_adt_2022`, but it is intentionally **not** converted into a fake live congestion speed.


In [ ]:
def cwwp_factor_color(factor):
    if factor is None:
        return "gray"

    f = float(factor)

    if f > 1.50:
        return "red"
    if f > 1.25:
        return "darkred"
    if f > 1.05:
        return "orange"

    return "green"


open_map = folium.Map(
    location=center,
    zoom_start=12,
    tiles=None,
)

folium.TileLayer(
    tiles=(
        "https://tile.openstreetmap.de/"
        "{z}/{x}/{y}.png"
    ),
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap.DE",
    overlay=False,
    control=False,
    max_zoom=18,
).add_to(open_map)


# ------------------------------------------------------------
# Irvine official Speed layer
# ------------------------------------------------------------
speed_group = folium.FeatureGroup(
    name="City of Irvine official Speed",
    show=False,
)

for seg in irvine_speed_segments:

    mph = seg.get(
        "value"
    )

    props = seg.get(
        "properties",
        {}
    )

    folium.PolyLine(
        seg["coords"],
        color="steelblue",
        weight=2,
        opacity=0.45,
        tooltip=(
            f"{props.get('STREETNAME')} | "
            f"{mph:.0f} mph"
        ),
        popup=(
            f"<b>{props.get('STREETNAME')}</b><br>"
            f"Official speed: {mph:.0f} mph<br>"
            f"Street category code: {props.get('CL')}"
        )
    ).add_to(
        speed_group
    )

speed_group.add_to(
    open_map
)


# ------------------------------------------------------------
# CWWP current corridor layer
# ------------------------------------------------------------
cwwp_group = folium.FeatureGroup(
    name="Caltrans CWWP2 current travel time",
    show=True,
)

for seg in cwwp_corridors:

    factor = seg.get(
        "factor"
    )

    observed_min = (
        float(
            seg.get(
                "observed_sec",
                0
            )
        )
        / 60.0
    )

    baseline_min = (
        float(
            seg.get(
                "baseline_sec",
                0
            )
        )
        / 60.0
    )

    popup = (
        f"<b>{seg.get('route')} "
        f"{seg.get('direction')}</b><br>"
        f"{seg.get('begin_name')} → "
        f"{seg.get('end_name')}<br>"
        f"Current: {observed_min:.1f} min<br>"
        f"Baseline: {baseline_min:.1f} min<br>"
        f"Factor: {factor:.2f}<br>"
        f"Timestamp: {seg.get('timestamp')}"
    )

    folium.PolyLine(
        seg["coords"],
        color=cwwp_factor_color(
            factor
        ),
        weight=5,
        opacity=0.80,
        tooltip=(
            f"{seg.get('route')} | "
            f"factor={factor:.2f}"
        ),
        popup=popup,
    ).add_to(
        cwwp_group
    )

cwwp_group.add_to(
    open_map
)


# ------------------------------------------------------------
# Customers / depot
# ------------------------------------------------------------
for _, row in customer_df.iterrows():

    if row[
        "customer_id"
    ] == "DEPOT":

        folium.Marker(
            [
                row["lat"],
                row["lon"]
            ],
            tooltip="DEPOT",
            icon=folium.Icon(
                color="black",
                icon="home",
                prefix="fa"
            ),
        ).add_to(
            open_map
        )

    else:
        folium.CircleMarker(
            [
                row["lat"],
                row["lon"]
            ],
            radius=4,
            color="blue",
            fill=True,
            fill_opacity=0.8,
            tooltip=row[
                "customer_id"
            ],
        ).add_to(
            open_map
        )


folium.LayerControl().add_to(
    open_map
)

open_map


# Chapter 3 — SAME Downstream Optimization as 07A1

**Talk-script anchor: B1-SAME-CORE**

**Skip detailed explanation.**

After the road-cost / traffic layer is created, the same core workflow continues:

**VRP → driver/time constraints → capacity checks → 3D packing → packing cuts → re-optimization**

## 11. 3D Packing feasibility checker

`py3dbp` receives:

- one truck cargo box
- every product assigned to that truck

The function tries two item-order heuristics.

We only ask:

> **Did every item fit?**

We do **not** optimize the packing arrangement itself.

In [ ]:
def get_products_for_customer_nodes(customer_nodes):
    """Return PHYSICAL boxes generated from the orders of these customers."""
    if not customer_nodes:
        return physical_item_df.iloc[0:0].copy()

    customer_ids = customer_df.loc[
        customer_nodes, "customer_id"
    ].tolist()

    return physical_item_df[
        physical_item_df["customer_id"].isin(customer_ids)
    ].copy()


def run_one_packing(vehicle_id, customer_nodes, bigger_first=True):
    truck = truck_df.loc[
        truck_df["vehicle_id"] == vehicle_id
    ].iloc[0]

    items = get_products_for_customer_nodes(customer_nodes)

    packer = Packer()

    # py3dbp order is width, height, depth.
    # depth is mapped to cargo-bed length.
    packer.add_bin(
        Bin(
            truck["truck_name"],
            int(truck["width_cm"]),
            int(truck["height_cm"]),
            int(truck["length_cm"]),
            int(truck["max_payload_kg"])
        )
    )

    for _, p in items.iterrows():
        packer.add_item(
            Item(
                p["item_id"],
                int(p["width_cm"]),
                int(p["height_cm"]),
                int(p["length_cm"]),
                int(p["weight_kg"])
            )
        )

    packer.pack(
        bigger_first=bigger_first,
        distribute_items=False,
        number_of_decimals=0
    )

    packed_bin = packer.bins[0]
    feasible = len(packed_bin.unfitted_items) == 0

    return {
        "feasible": feasible,
        "bin": packed_bin,
        "items": items,
        "unfitted": [x.name for x in packed_bin.unfitted_items],
        "bigger_first": bigger_first
    }


def check_3d_packing(vehicle_id, customer_nodes):
    # Feasibility only: try two deterministic item-order heuristics.
    attempts = [
        run_one_packing(vehicle_id, customer_nodes, bigger_first=True),
        run_one_packing(vehicle_id, customer_nodes, bigger_first=False)
    ]

    feasible_attempts = [
        x for x in attempts if x["feasible"]
    ]

    if feasible_attempts:
        return feasible_attempts[0]

    return attempts[0]

## 12. Quick geometric sanity check — which customers cannot fit the Small Truck by themselves?

In [ ]:
# Truck 3 is used as the representative Small Truck here because
# Trucks 3, 4, and 5 have identical cargo-bed dimensions/capacity.
small_vehicle_id = 2

single_customer_check = []

for node in range(1, len(customer_df)):
    result = check_3d_packing(small_vehicle_id, [node])

    single_customer_check.append({
        "customer_id": customer_df.loc[node, "customer_id"],
        "customer_name": customer_df.loc[node, "customer_name"],
        "small_truck_fit": result["feasible"],
        "unfitted_products": ", ".join(result["unfitted"])
    })

single_customer_check_df = pd.DataFrame(single_customer_check)

print(
    "Representative Small Truck:",
    truck_df.loc[
        truck_df["vehicle_id"] == small_vehicle_id,
        "truck_name"
    ].iloc[0]
)
print("Truck 3/4/5 share the same Small Truck specification.")

display(
    single_customer_check_df[
        single_customer_check_df["small_truck_fit"] == False
    ]
)

## 13. VRP solver with iterative packing cuts

Before the VRP, driver qualification and staffing determine which physical trucks can operate.

The VRP keeps these as **hard constraints**:

1. Customer delivery time windows
2. Truck payload capacity
3. Aggregate cargo volume
4. Driver/truck qualification and truck availability
5. Driver route departure cannot be before shift start
6. 3D physical packability through iterative packing cuts

The scheduled **driver shift end is soft**:

> A route may return late when necessary, but overtime receives a large objective penalty.

The objective therefore balances:

```text
traffic-aware travel time
+
large penalty × total overtime minutes
```

This strongly prefers routes that finish inside scheduled shifts, but it does not reject an otherwise feasible delivery plan solely because a driver returns late.

Packing cuts are added only after a 3D Packing failure.

A cut means:

> “Do not put every customer in this conflict set on this same truck again.”


### 07B1 compatibility note — OR-Tools vehicle filter

`SetAllowedVehiclesForIndex()` is avoided because some current OR-Tools Python builds can fail while converting the Python vehicle list to `absl::Span<int const>`.

The model keeps the same staffing rule by explicitly forbidding only unavailable trucks through `VehicleVar(index) != vehicle_id` constraints. Driver shifts, truck qualifications, customer time windows, capacity constraints, traffic costs, and 3D-packing cuts remain unchanged.


### 07B1 — Soft driver overtime objective

The scheduled driver shift end is a soft route limit. OR-Tools may return a truck after shift end, but each overtime minute receives a large objective penalty.

This makes the solver minimize **total overtime across the fleet** while keeping customer windows, capacities, qualification, and 3D packing as hard constraints.


In [ ]:
NUM_VEHICLES = len(truck_df)

weight_demands = customer_df["weight_kg"].astype(int).tolist()
volume_demands = customer_df["volume_liters"].astype(int).tolist()
service_minutes = customer_df["service_min"].astype(int).tolist()

# Customer source windows are stored as minutes from 08:00.
# Convert them to the optimizer clock, where 0 = 06:00.
time_windows = [
    (
        int(start) + CUSTOMER_TO_SCHEDULE_OFFSET_MIN,
        int(end) + CUSTOMER_TO_SCHEDULE_OFFSET_MIN,
    )
    for start, end in zip(
        customer_df["window_start"].astype(int),
        customer_df["window_end"].astype(int),
    )
]

driver_shift_starts = {
    int(row["vehicle_id"]): int(row["shift_start_min"])
    for _, row in vehicle_driver_df[
        vehicle_driver_df["available"].fillna(False)
    ].iterrows()
}

driver_shift_ends = {
    int(row["vehicle_id"]): int(row["shift_end_min"])
    for _, row in vehicle_driver_df[
        vehicle_driver_df["available"].fillna(False)
    ].iterrows()
}


weight_capacities = (
    truck_df.sort_values("vehicle_id")["max_payload_kg"]
    .astype(int)
    .tolist()
)

volume_capacities = (
    truck_df.sort_values("vehicle_id")["cargo_volume_liters"]
    .round()
    .astype(int)
    .tolist()
)

# ============================================================
# DRIVER OVERTIME POLICY
# ============================================================
# Shift start remains hard.
# Shift end is soft: every minute after shift end adds this
# coefficient to the routing objective.
#
# Road travel uses roughly 1 objective unit per travel minute.
# A value of 100 therefore makes avoiding overtime much more
# important than shaving a few minutes from the route.
OVERTIME_PENALTY_PER_MIN = 100

# Safety horizon only. This is NOT the preferred shift extension.
# It simply prevents the time dimension from being unbounded.
# Scheduled shifts end by 18:00 in this demo; allowing 360 extra
# minutes means the model can still return as late as midnight
# if the business constraints truly require it.
MAX_OVERTIME_MIN = 360
OVERTIME_HORIZON_MIN = (
    SCHEDULE_HORIZON_MIN + MAX_OVERTIME_MIN
)


def solve_vrp_with_packing_cuts(
    packing_cuts,
    solve_seconds=10,
    depot=0
):
    n = len(road_time_min)

    manager = pywrapcp.RoutingIndexManager(
        n,
        NUM_VEHICLES,
        depot
    )
    routing = pywrapcp.RoutingModel(manager)

    # --------------------------------------------------------------
    # Primary objective: road travel time
    # --------------------------------------------------------------
    def travel_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return int(road_time_min[from_node][to_node])

    travel_cb = routing.RegisterTransitCallback(travel_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(travel_cb)

    # --------------------------------------------------------------
    # Weight capacity
    # --------------------------------------------------------------
    def weight_callback(from_index):
        node = manager.IndexToNode(from_index)
        return int(weight_demands[node])

    weight_cb = routing.RegisterUnaryTransitCallback(weight_callback)

    routing.AddDimensionWithVehicleCapacity(
        weight_cb,
        0,
        weight_capacities,
        True,
        "Weight"
    )

    # --------------------------------------------------------------
    # Aggregate volume capacity
    # This is only a fast pre-filter. 3D packing is checked later.
    # --------------------------------------------------------------
    def volume_callback(from_index):
        node = manager.IndexToNode(from_index)
        return int(volume_demands[node])

    volume_cb = routing.RegisterUnaryTransitCallback(volume_callback)

    routing.AddDimensionWithVehicleCapacity(
        volume_cb,
        0,
        volume_capacities,
        True,
        "Volume"
    )

    # --------------------------------------------------------------
    # Time windows + customer service times
    # --------------------------------------------------------------
    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        return (
            int(road_time_min[from_node][to_node])
            + int(service_minutes[from_node])
        )

    time_cb = routing.RegisterTransitCallback(time_callback)

    routing.AddDimension(
        time_cb,
        OVERTIME_HORIZON_MIN,  # waiting time allowed
        OVERTIME_HORIZON_MIN,  # hard safety horizon; shifts themselves are soft-ended
        False,                 # each driver has a different start range
        "Time"
    )

    time_dim = routing.GetDimensionOrDie("Time")
    solver = routing.solver()

    # --------------------------------------------------------------
    # Pre-dispatch staffing filter
    # --------------------------------------------------------------
    # Do NOT use RoutingModel.SetAllowedVehiclesForIndex() here.
    #
    # Recent OR-Tools Python builds can raise:
    #   TypeError: argument 2 of type 'absl::Span< int const >'
    # even when a normal Python list[int] is supplied.
    #
    # Instead, explicitly forbid only unavailable physical trucks
    # with ordinary CP constraints on VehicleVar(index). This keeps
    # the same model semantics and avoids the binding problem.
    available_vehicle_set = {
        int(v) for v in available_vehicle_ids
    }
    unavailable_vehicle_ids = [
        vehicle_id
        for vehicle_id in range(NUM_VEHICLES)
        if vehicle_id not in available_vehicle_set
    ]

    for node in range(1, n):
        idx = manager.NodeToIndex(node)
        start, end = time_windows[node]
        time_dim.CumulVar(idx).SetRange(int(start), int(end))

        for vehicle_id in unavailable_vehicle_ids:
            solver.Add(
                routing.VehicleVar(idx) != int(vehicle_id)
            )

    # ----------------------------------------------------------
    # Driver working-shift constraints with SOFT OVERTIME
    #
    # HARD:
    #   route cannot start before the driver's shift begins.
    #
    # SOFT:
    #   route may return after the scheduled shift end, but every
    #   overtime minute adds OVERTIME_PENALTY_PER_MIN to the
    #   objective. Because this is applied to every vehicle end,
    #   the optimizer minimizes TOTAL overtime across drivers.
    # ----------------------------------------------------------
    for vehicle_id in range(NUM_VEHICLES):
        start_var = time_dim.CumulVar(
            routing.Start(vehicle_id)
        )
        end_var = time_dim.CumulVar(
            routing.End(vehicle_id)
        )

        if vehicle_id in available_vehicle_set:
            shift_start = int(
                driver_shift_starts[vehicle_id]
            )
            shift_end = int(
                driver_shift_ends[vehicle_id]
            )

            # A driver cannot begin work before the scheduled start.
            # Starting after shift end is also prevented.
            start_var.SetRange(
                shift_start,
                shift_end
            )

            # Returning late is allowed up to the broad safety horizon.
            end_var.SetRange(
                shift_start,
                OVERTIME_HORIZON_MIN
            )

            # Soft upper bound = scheduled shift end.
            # Objective penalty:
            #   max(0, depot_return - shift_end)
            #   * OVERTIME_PENALTY_PER_MIN
            time_dim.SetCumulVarSoftUpperBound(
                routing.End(vehicle_id),
                shift_end,
                int(OVERTIME_PENALTY_PER_MIN)
            )
        else:
            # Unstaffed / unqualified truck:
            # force an idle depot-to-depot route at schedule zero.
            start_var.SetRange(0, 0)
            end_var.SetRange(0, 0)

        routing.AddVariableMinimizedByFinalizer(
            start_var
        )
        routing.AddVariableMinimizedByFinalizer(
            end_var
        )

    # --------------------------------------------------------------
    # Packing feedback cuts
    # --------------------------------------------------------------

    for cut_id, cut in enumerate(packing_cuts):
        vehicle_id = int(cut["vehicle_id"])
        nodes = list(cut["nodes"])

        on_this_vehicle = []

        for node in nodes:
            idx = manager.NodeToIndex(int(node))
            on_this_vehicle.append(
                solver.IsEqualCstVar(
                    routing.VehicleVar(idx),
                    vehicle_id
                )
            )

        # Not all nodes in this conflict set may be assigned
        # to this same truck again.
        solver.Add(
            solver.Sum(on_this_vehicle) <= len(nodes) - 1
        )

    # --------------------------------------------------------------
    # Search
    # --------------------------------------------------------------
    search = pywrapcp.DefaultRoutingSearchParameters()

    search.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    )

    search.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )

    search.time_limit.seconds = solve_seconds

    solution = routing.SolveWithParameters(search)

    return manager, routing, solution

## 14. Fast feasibility pre-check after staffing

Before asking OR-Tools to solve the complete routing problem, run a quick per-customer check.

This is **not** the final proof of feasibility. It only asks whether each customer has at least one currently staffed truck that:

- is available during part of the customer's delivery window,
- can carry that customer's aggregate weight, and
- can carry that customer's aggregate volume.

Road travel, route interactions, and exact 3D packing are still checked later.


In [ ]:
def build_staffing_feasibility_precheck():
    rows = []

    available_rows = vehicle_driver_df[
        vehicle_driver_df["available"].fillna(False)
    ]

    for node in range(1, len(customer_df)):
        customer = customer_df.loc[node]

        window_start = (
            int(customer["window_start"])
            + CUSTOMER_TO_SCHEDULE_OFFSET_MIN
        )
        window_end = (
            int(customer["window_end"])
            + CUSTOMER_TO_SCHEDULE_OFFSET_MIN
        )

        regular_candidates = []
        overtime_candidates = []

        for _, vt in available_rows.iterrows():
            shift_start = int(vt["shift_start_min"])
            shift_end = int(vt["shift_end_min"])

            weight_ok = (
                int(customer["weight_kg"])
                <= int(vt["max_payload_kg"])
            )

            volume_ok = (
                int(customer["volume_liters"])
                <= int(round(vt["cargo_volume_liters"]))
            )

            if not (weight_ok and volume_ok):
                continue

            # Fast individual check only:
            # can the truck arrive during the customer window at
            # some point after the driver's shift begins?
            earliest_possible = max(
                window_start,
                shift_start,
            )

            if earliest_possible > window_end:
                continue

            label = (
                f"{vt['truck_name']} / {vt['driver_name']}"
            )

            if earliest_possible <= shift_end:
                regular_candidates.append(label)
            elif earliest_possible <= OVERTIME_HORIZON_MIN:
                overtime_candidates.append(label)

        all_candidates = (
            regular_candidates + overtime_candidates
        )

        rows.append({
            "customer_id": customer["customer_id"],
            "window": customer["time_window"],
            "weight_kg": int(customer["weight_kg"]),
            "volume_liters": int(customer["volume_liters"]),
            "regular_shift_candidates": len(regular_candidates),
            "overtime_only_candidates": len(overtime_candidates),
            "candidate_truck_driver_pairs": len(all_candidates),
            "regular_candidates": ", ".join(regular_candidates),
            "overtime_candidates": ", ".join(overtime_candidates),
            "precheck": (
                "PASS"
                if regular_candidates
                else (
                    "OVERTIME POSSIBLE"
                    if overtime_candidates
                    else "NO CANDIDATE"
                )
            ),
        })

    return pd.DataFrame(rows)


staffing_precheck_df = (
    build_staffing_feasibility_precheck()
)

display(staffing_precheck_df)

no_candidate_customers = staffing_precheck_df[
    staffing_precheck_df["precheck"] == "NO CANDIDATE"
]

overtime_only_customers = staffing_precheck_df[
    staffing_precheck_df["precheck"] == "OVERTIME POSSIBLE"
]

if len(no_candidate_customers) == 0:
    print(
        "PRE-CHECK PASS: every customer has at least one "
        "staffed truck/driver candidate."
    )

    if len(overtime_only_customers):
        print(
            "NOTE: some individual customers may require overtime "
            "under the current roster:"
        )
        display(overtime_only_customers)

    print(
        "The full VRP will now optimize traffic, time windows, "
        "capacities, total driver overtime, and 3D packing."
    )
else:
    print(
        "PRE-CHECK WARNING: some customers have no obvious "
        "truck/driver candidate even with overtime:"
    )
    display(no_candidate_customers)

    print(
        "Possible operational actions: request a delivery-window "
        "change, call an additional qualified driver, or add "
        "another suitable truck."
    )


def print_infeasibility_guidance():
    print()
    print("=" * 80)
    print("NO FEASIBLE DELIVERY PLAN")
    print("=" * 80)

    blocked = staffing_precheck_df[
        staffing_precheck_df["precheck"] == "NO CANDIDATE"
    ]

    if len(blocked):
        print(
            "Some customers already fail the staffing/capacity "
            "pre-check even though overtime is allowed:"
        )
        display(blocked)
    else:
        print(
            "All customers pass the simple individual pre-check, "
            "so infeasibility comes from the combined hard constraints."
        )
        print(
            "Possible causes include customer time windows, route "
            "travel time, truck qualification/availability, payload, "
            "aggregate volume, the broad safety horizon, or 3D "
            "packing conflicts."
        )

    print()
    print("Operational options:")
    print("  - Ask affected customers to change delivery windows")
    print("  - Call an additional qualified / emergency driver")
    print("  - Swap driver shifts or route assignments")
    print("  - Add another suitable truck")
    print("  - Split or reschedule deliveries")


## 14. Extract routes, arrival times, truck loads, and depot return times

In [ ]:
def extract_vrp_solution(manager, routing, solution):
    time_dim = routing.GetDimensionOrDie("Time")

    route_rows = []
    stop_rows = []

    for vehicle_id in range(NUM_VEHICLES):
        index = routing.Start(vehicle_id)

        start_min = solution.Min(
            time_dim.CumulVar(index)
        )

        driver_row = vehicle_driver_df.loc[
            vehicle_driver_df["vehicle_id"] == vehicle_id
        ].iloc[0]

        driver_name = (
            driver_row["driver_name"]
            if pd.notna(driver_row["driver_name"])
            else "NO DRIVER"
        )

        driver_shift = (
            f"{driver_row['shift_start']}-{driver_row['shift_end']}"
            if pd.notna(driver_row["shift_start"])
            else "UNAVAILABLE"
        )

        route_nodes = []
        travel_total = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route_nodes.append(node)

            next_index = solution.Value(routing.NextVar(index))
            next_node = manager.IndexToNode(next_index)

            travel_total += int(
                road_time_min[node][next_node]
            )

            if node != 0:
                arrival = solution.Min(
                    time_dim.CumulVar(index)
                )

                stop_rows.append({
                    "vehicle_id": vehicle_id,
                    "truck": truck_df.loc[
                        truck_df.vehicle_id == vehicle_id,
                        "truck_name"
                    ].iloc[0],
                    "driver": driver_name,
                    "driver_shift": driver_shift,
                    "customer_id": customer_df.loc[
                        node, "customer_id"
                    ],
                    "arrival": hhmm(arrival),
                    "window": customer_df.loc[
                        node, "time_window"
                    ],
                    "service_min": int(
                        customer_df.loc[node, "service_min"]
                    ),
                    "weight_kg": int(
                        customer_df.loc[node, "weight_kg"]
                    ),
                    "volume_liters": int(
                        customer_df.loc[node, "volume_liters"]
                    )
                })

            index = next_index

        route_nodes.append(0)

        finish_min = solution.Min(
            time_dim.CumulVar(index)
        )

        customer_nodes = [
            n for n in route_nodes if n != 0
        ]

        if pd.notna(driver_row["shift_end_min"]):
            scheduled_shift_end_min = int(
                driver_row["shift_end_min"]
            )
            overtime_min = max(
                0,
                int(finish_min) - scheduled_shift_end_min
            )
            shift_slack_min = max(
                0,
                scheduled_shift_end_min - int(finish_min)
            )
        else:
            scheduled_shift_end_min = None
            overtime_min = 0
            shift_slack_min = 0

        shift_status = (
            f"OVERTIME +{overtime_min} min"
            if overtime_min > 0
            else "WITHIN SHIFT"
        )

        route_rows.append({
            "vehicle_id": vehicle_id,
            "truck": truck_df.loc[
                truck_df.vehicle_id == vehicle_id,
                "truck_name"
            ].iloc[0],
            "truck_type": truck_df.loc[
                truck_df.vehicle_id == vehicle_id,
                "truck_type"
            ].iloc[0],
            "driver": driver_name,
            "driver_shift": driver_shift,
            "route_nodes": route_nodes,
            "route": " -> ".join(
                customer_df.loc[n, "customer_id"]
                for n in route_nodes
            ),
            "customers": len(customer_nodes),
            "status": "ACTIVE" if len(customer_nodes) > 0 else "IDLE",
            "weight_kg": int(
                customer_df.loc[
                    customer_nodes, "weight_kg"
                ].sum()
            ),
            "weight_capacity_kg": int(
                truck_df.loc[
                    truck_df.vehicle_id == vehicle_id,
                    "max_payload_kg"
                ].iloc[0]
            ),
            "volume_liters": int(
                customer_df.loc[
                    customer_nodes, "volume_liters"
                ].sum()
            ),
            "volume_capacity_liters": int(
                round(
                    truck_df.loc[
                        truck_df.vehicle_id == vehicle_id,
                        "cargo_volume_liters"
                    ].iloc[0]
                )
            ),
            "travel_min": travel_total,
            "start_min": start_min,
            "depot_departure": hhmm(start_min),
            "finish_min": finish_min,
            "depot_return": hhmm(finish_min),
            "scheduled_shift_end_min": scheduled_shift_end_min,
            "overtime_min": int(overtime_min),
            "shift_slack_min": int(shift_slack_min),
            "shift_status": shift_status,
            "overtime_objective_penalty": int(
                overtime_min * OVERTIME_PENALTY_PER_MIN
            )
        })

    return (
        pd.DataFrame(route_rows),
        pd.DataFrame(stop_rows)
    )

## 15. Conflict-set reduction

When a truck cannot be packed, we do not immediately ban every customer in its route.

We first check:

1. Does any single customer's shipment fail by itself?
2. If not, can we shrink the failing customer set while it remains infeasible?

The resulting set is a small **packing conflict set** returned to the VRP.

In [ ]:
def reduce_packing_conflict(vehicle_id, customer_nodes):
    customer_nodes = list(customer_nodes)

    # First look for a direct one-customer geometric incompatibility.
    for node in customer_nodes:
        if not check_3d_packing(
            vehicle_id,
            [node]
        )["feasible"]:
            return [node]

    # Greedy inclusion-minimal reduction.
    conflict = list(customer_nodes)

    changed = True

    while changed and len(conflict) > 1:
        changed = False

        for node in list(conflict):
            trial = [
                x for x in conflict
                if x != node
            ]

            if not trial:
                continue

            if not check_3d_packing(
                vehicle_id,
                trial
            )["feasible"]:
                conflict = trial
                changed = True
                break

    return conflict


def cut_signature(cut):
    return (
        int(cut["vehicle_id"]),
        tuple(sorted(int(x) for x in cut["nodes"]))
    )

## 16. Run: VRP → 3D Packing → cuts → VRP again

In [ ]:
MAX_ITERATIONS = 20
VRP_SOLVE_SECONDS = 10

packing_cuts = []
iteration_history = []
final_routes = None
final_stops = None
final_packings = None

for iteration in range(1, MAX_ITERATIONS + 1):

    print("=" * 80)
    print(f"ITERATION {iteration}")
    print("Existing packing cuts:", len(packing_cuts))

    manager, routing, solution = solve_vrp_with_packing_cuts(
        packing_cuts,
        solve_seconds=VRP_SOLVE_SECONDS
    )

    if solution is None:
        print_infeasibility_guidance()
        raise RuntimeError(
            "VRP is infeasible under the current "
            "driver/truck/customer/packing constraints."
        )

    routes_df, stops_df = extract_vrp_solution(
        manager,
        routing,
        solution
    )

    display(routes_df[[
        "truck",
        "truck_type",
        "driver",
        "driver_shift",
        "status",
        "depot_departure",
        "route",
        "weight_kg",
        "weight_capacity_kg",
        "volume_liters",
        "volume_capacity_liters",
        "travel_min",
        "depot_return",
        "shift_status",
        "overtime_min",
    ]])

    total_overtime_min = int(
        routes_df["overtime_min"].sum()
    )

    print(
        "Total driver overtime:",
        total_overtime_min,
        "min",
        "| penalty coefficient:",
        OVERTIME_PENALTY_PER_MIN,
        "per min"
    )

    overtime_rows = routes_df[
        routes_df["overtime_min"] > 0
    ]

    if len(overtime_rows):
        for _, ot in overtime_rows.iterrows():
            print(
                f"  WARNING: {ot['driver']} on {ot['truck']} "
                f"has {int(ot['overtime_min'])} min overtime "
                f"(return {ot['depot_return']}, "
                f"shift {ot['driver_shift']})."
            )

    all_fit = True
    iteration_packings = {}
    new_cuts = []

    for _, route_row in routes_df.iterrows():
        vehicle_id = int(route_row["vehicle_id"])

        customer_nodes = [
            n for n in route_row["route_nodes"]
            if n != 0
        ]

        pack_result = check_3d_packing(
            vehicle_id,
            customer_nodes
        )

        iteration_packings[vehicle_id] = pack_result

        status = "FIT" if pack_result["feasible"] else "NOT FIT"

        print(
            f"{route_row['truck']} ({route_row['truck_type']}): "
            f"{status} | "
            f"{len(pack_result['items'])} products | "
            f"{route_row['weight_kg']} kg"
        )

        if not pack_result["feasible"]:
            all_fit = False

            conflict_nodes = reduce_packing_conflict(
                vehicle_id,
                customer_nodes
            )

            conflict_customers = [
                customer_df.loc[n, "customer_id"]
                for n in conflict_nodes
            ]

            # --------------------------------------------------
            # Propagate this packing conflict to EVERY currently
            # available physical truck on which the same customer
            # set is also not 3D-packable.
            #
            # Without this step, two equivalent trucks can simply
            # swap the same bad customer set back and forth.
            # --------------------------------------------------
            existing_signatures = {
                cut_signature(c)
                for c in packing_cuts + new_cuts
            }

            cut_vehicles = []

            for target_vehicle_id in available_vehicle_ids:
                target_vehicle_id = int(target_vehicle_id)

                target_check = check_3d_packing(
                    target_vehicle_id,
                    conflict_nodes
                )

                if not target_check["feasible"]:
                    candidate_cut = {
                        "vehicle_id": target_vehicle_id,
                        "nodes": list(conflict_nodes),
                    }

                    sig = cut_signature(candidate_cut)

                    if sig not in existing_signatures:
                        new_cuts.append(candidate_cut)
                        existing_signatures.add(sig)

                    cut_vehicles.append(target_vehicle_id)

            print(
                "  Packing conflict:",
                conflict_customers
            )
            print(
                "  Cut applied to truck IDs:",
                cut_vehicles
            )

            if pack_result["unfitted"]:
                print(
                    "  Unfitted products:",
                    pack_result["unfitted"]
                )

    iteration_history.append({
        "iteration": iteration,
        "total_travel_min": int(routes_df["travel_min"].sum()),
        "total_overtime_min": int(routes_df["overtime_min"].sum()),
        "overtime_penalty": int(
            routes_df["overtime_objective_penalty"].sum()
        ),
        "all_fit": all_fit,
        "cuts_before": len(packing_cuts),
        "new_cuts": len(new_cuts)
    })

    if all_fit:
        final_routes = routes_df.copy()
        final_stops = stops_df.copy()
        final_packings = iteration_packings

        print()
        print("FINAL RESULT: all truck loads are 3D-packable.")
        break

    if not new_cuts:
        raise RuntimeError(
            "Packing is infeasible but no new cut could be generated."
        )

    packing_cuts.extend(new_cuts)

else:
    raise RuntimeError(
        f"No fully packable solution found after {MAX_ITERATIONS} iterations."
    )

# Chapter 4 — Result with the Public / Open Data Road Model

**Talk-script anchor: B1-RESULTS**

Briefly show the final iteration history, routes, overtime, packing result, and route map.

The emphasis is not that public data is identical to TomTom; the emphasis is that the **same downstream optimizer can consume a different road-cost data layer**.

## 17. Iteration history

In [ ]:
iteration_history_df = pd.DataFrame(iteration_history)
display(iteration_history_df)

print("Final number of packing cuts:", len(packing_cuts))

if packing_cuts:
    readable_cuts = []

    for i, cut in enumerate(packing_cuts, start=1):
        readable_cuts.append({
            "cut": i,
            "truck": truck_df.loc[
                truck_df.vehicle_id == cut["vehicle_id"],
                "truck_name"
            ].iloc[0],
            "customers": ", ".join(
                customer_df.loc[n, "customer_id"]
                for n in cut["nodes"]
            )
        })

    display(pd.DataFrame(readable_cuts))

## 18. Final routing solution

In [ ]:
display(final_routes[[
    "truck", "truck_type",
    "driver", "driver_shift",
    "status", "depot_departure",
    "route", "customers",
    "weight_kg", "weight_capacity_kg",
    "volume_liters", "volume_capacity_liters",
    "travel_min", "depot_return",
    "shift_status", "overtime_min",
    "shift_slack_min"
]])

display(
    final_stops
    .sort_values(["vehicle_id", "arrival"])
    .reset_index(drop=True)
)

print(
    "Total travel minutes:",
    int(final_routes["travel_min"].sum())
)

total_overtime_min = int(
    final_routes["overtime_min"].sum()
)

print(
    "Total driver overtime:",
    total_overtime_min,
    "minutes"
)

active_trucks = int(
    (final_routes["status"] == "ACTIVE").sum()
)
idle_trucks = int(
    (final_routes["status"] == "IDLE").sum()
)

print("Active trucks:", active_trucks, "/", NUM_VEHICLES)
print("Idle trucks:", idle_trucks)

print()
print("=" * 80)

overtime_final = (
    final_routes[
        final_routes["overtime_min"] > 0
    ][[
        "truck",
        "driver",
        "driver_shift",
        "depot_return",
        "overtime_min",
        "route",
    ]]
    .sort_values(
        ["overtime_min", "driver"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

if len(overtime_final) == 0:
    print(
        "SHIFT STATUS: all drivers return within their scheduled shifts."
    )
else:
    print(
        "OVERTIME WARNING: the optimizer found the lowest-overtime "
        "plan it could within the current objective and constraints."
    )
    display(overtime_final)

    for _, row in overtime_final.iterrows():
        print(
            f"  -> {row['driver']}: +{int(row['overtime_min'])} min "
            f"on {row['truck']} "
            f"(scheduled {row['driver_shift']}, "
            f"return {row['depot_return']})"
        )

    print()
    print(
        "Dispatcher action: consider a shift swap, route reassignment, "
        "additional qualified driver, or accept the overtime."
    )

print("=" * 80)


## 19. Final 3D packing summary

In [ ]:
packing_summary = []

for _, route_row in final_routes.iterrows():
    vehicle_id = int(route_row["vehicle_id"])
    result = final_packings[vehicle_id]

    customer_nodes = [
        n for n in route_row["route_nodes"]
        if n != 0
    ]

    items = get_products_for_customer_nodes(
        customer_nodes
    )

    truck = truck_df.loc[
        truck_df.vehicle_id == vehicle_id
    ].iloc[0]

    packing_summary.append({
        "truck": truck["truck_name"],
        "truck_type": truck["truck_type"],
        "status": route_row["status"],
        "customers": ", ".join(
            customer_df.loc[n, "customer_id"]
            for n in customer_nodes
        ),
        "physical_boxes": len(items),
        "packing": "FIT" if result["feasible"] else "NOT FIT",
        "load_weight_kg": int(items["weight_kg"].sum()),
        "payload_capacity_kg": int(truck["max_payload_kg"]),
        "load_volume_m3": round(
            items["volume_liters"].sum() / 1000,
            2
        ),
        "bed_volume_m3": round(
            truck["cargo_volume_m3"],
            2
        ),
        "packing_heuristic": (
            "bigger_first"
            if result["bigger_first"]
            else "smaller_first"
        )
    })

packing_summary_df = pd.DataFrame(packing_summary)
display(packing_summary_df)

## 20. Visualize final routes — different color for each truck + warehouse

In [ ]:
route_map = folium.Map(
    location=center,
    zoom_start=11,
    tiles=None
)

folium.TileLayer(
    tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap.DE",
    overlay=False,
    control=False,
    max_zoom=18
).add_to(route_map)

truck_colors = {
    0: "blue",
    1: "red",
    2: "green",
    3: "purple",
    4: "orange"
}

# Warehouse / depot
folium.Marker(
    [
        customer_df.loc[0, "lat"],
        customer_df.loc[0, "lon"]
    ],
    tooltip="DEPOT / WAREHOUSE",
    popup=(
        "<b>DEPOT / WAREHOUSE</b><br>"
        "Truck operating times depend on assigned drivers<br>"
        "See route table for each driver shift"
    ),
    icon=folium.Icon(
        color="black",
        icon="home",
        prefix="fa"
    )
).add_to(route_map)

# Route lines
for _, route_row in final_routes.iterrows():
    vehicle_id = int(route_row["vehicle_id"])
    route_nodes = route_row["route_nodes"]

    for a, b in zip(
        route_nodes[:-1],
        route_nodes[1:]
    ):
        if G is not None and osm_nodes is not None:
            try:
                path = nx.shortest_path(
                    G,
                    osm_nodes[a],
                    osm_nodes[b],
                    weight="traffic_travel_time"
                )

                coordinates = [
                    (
                        G.nodes[node]["y"],
                        G.nodes[node]["x"]
                    )
                    for node in path
                ]

            except Exception:
                coordinates = [
                    (
                        customer_df.loc[a, "lat"],
                        customer_df.loc[a, "lon"]
                    ),
                    (
                        customer_df.loc[b, "lat"],
                        customer_df.loc[b, "lon"]
                    )
                ]

        else:
            coordinates = [
                (
                    customer_df.loc[a, "lat"],
                    customer_df.loc[a, "lon"]
                ),
                (
                    customer_df.loc[b, "lat"],
                    customer_df.loc[b, "lon"]
                )
            ]

        folium.PolyLine(
            coordinates,
            color=truck_colors[vehicle_id],
            weight=5,
            opacity=0.85,
            tooltip=route_row["truck"]
        ).add_to(route_map)

# Optional Caltrans CWWP2 current-traffic overlay on the final route map
if cwwp_corridors:
    traffic_overlay = folium.FeatureGroup(
        name="Caltrans CWWP2 current corridors",
        show=False,
    )

    for seg in cwwp_corridors:
        factor = seg.get("factor")

        folium.PolyLine(
            seg["coords"],
            color=cwwp_factor_color(
                factor
            ),
            weight=3,
            opacity=0.50,
            tooltip=(
                f"{seg.get('route')} | "
                f"current factor={factor:.2f}"
            ),
        ).add_to(
            traffic_overlay
        )

    traffic_overlay.add_to(
        route_map
    )

# Customer markers
for _, stop in final_stops.iterrows():
    node = int(
        customer_df.index[
            customer_df["customer_id"]
            == stop["customer_id"]
        ][0]
    )

    vehicle_id = int(stop["vehicle_id"])

    popup = (
        f"<b>{stop.customer_id}</b><br>"
        f"{stop.truck}<br>"
        f"Driver: {stop.driver} ({stop.driver_shift})<br>"
        f"Arrival: {stop.arrival}<br>"
        f"Window: {stop.window}<br>"
        f"Service: {stop.service_min} min<br>"
        f"Weight: {stop.weight_kg} kg<br>"
        f"Volume: {stop.volume_liters} L"
    )

    folium.CircleMarker(
        [
            customer_df.loc[node, "lat"],
            customer_df.loc[node, "lon"]
        ],
        radius=7,
        color=truck_colors[vehicle_id],
        fill=True,
        fill_color=truck_colors[vehicle_id],
        fill_opacity=0.9,
        tooltip=f"{stop.customer_id} / {stop.truck}",
        popup=popup
    ).add_to(route_map)

legend_html = """
<div style="
position: fixed;
bottom: 35px;
left: 35px;
z-index: 9999;
background-color: white;
border: 2px solid #777;
border-radius: 6px;
padding: 10px 14px;
font-size: 14px;">
<b>Final Routes</b><br>
<span style="color:blue;">■</span> Truck 1 — Large<br>
<span style="color:red;">■</span> Truck 2 — Large<br>
<span style="color:green;">■</span> Truck 3 — Small<br>
<span style="color:purple;">■</span> Truck 4 — Small<br>
<span style="color:orange;">■</span> Truck 5 — Small<br>
<span style="color:black;">★</span> Depot / Warehouse
</div>
"""

route_map.get_root().html.add_child(
    folium.Element(legend_html)
)

route_map

## 21. Optional: simple 3D view of the packed boxes

This plot is for presentation/debugging only.

The **packing feasibility decision** comes from `py3dbp`, not from the plot.

In [ ]:
def plot_packed_truck(vehicle_id):
    result = final_packings[vehicle_id]
    packed_bin = result["bin"]

    truck = truck_df.loc[
        truck_df.vehicle_id == vehicle_id
    ].iloc[0]

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection="3d")

    for item in packed_bin.items:
        x, y, z = [float(v) for v in item.position]
        dx, dy, dz = [
            float(v)
            for v in item.get_dimension()
        ]

        ax.bar3d(
            x, y, z,
            dx, dy, dz,
            alpha=0.35,
            shade=True
        )

        ax.text(
            x + dx / 2,
            y + dy / 2,
            z + dz / 2,
            item.name,
            fontsize=7
        )

    ax.set_xlim(0, float(truck["width_cm"]))
    ax.set_ylim(0, float(truck["height_cm"]))
    ax.set_zlim(0, float(truck["length_cm"]))

    ax.set_xlabel("Width (cm)")
    ax.set_ylabel("Height (cm)")
    ax.set_zlabel("Length (cm)")

    ax.set_title(
        f"{truck['truck_name']} ({truck['truck_type']}) — 3D Packing"
    )

    plt.show()


for vehicle_id in range(NUM_VEHICLES):
    plot_packed_truck(vehicle_id)

# Chapter 5 — Open-Data Interpretation and Limitations

**Talk-script anchor: B1-SUMMARY**

Use this section to explain the practical trade-off:

- commercial traffic data: broader / more consistent integration
- public data: potentially low-cost, but regional and less standardized

## 22. What this version adds

### Open/public traffic data + workforce-aware optimization

This notebook keeps the **same driver, VRP, and 3D-packing logic as 07A1**, but replaces the commercial TomTom traffic layer with open/public sources.

The road model combines:

1. **OpenStreetMap / OSMnx**
   - routable road geometry
2. **City of Irvine official Speed + ADT**
   - official local/arterial baseline information
   - ADT is traffic-volume context, not live speed
3. **Caltrans CWWP2 District 12**
   - current highway travel-time observations

The semantic distinction is important:

```text
Irvine Speed
    = official baseline road speed

Irvine ADT
    = traffic-volume context

CWWP2
    = current dynamic highway travel time
```

The optimizer then uses:

```text
traffic_travel_time(edge)
```

and OR-Tools chooses truck assignment, customer visit order, and route timing while respecting:

- customer delivery time windows
- service time
- payload weight
- aggregate cargo volume
- driver/truck qualification
- hard driver shift start
- **soft driver shift end**
- iterative 3D packability constraints

### Driver overtime remains a soft exception

A driver may return after the scheduled shift end if necessary.

```text
overtime minutes
    ↓
large objective penalty
    ↓
optimizer tries to move work to other trucks/drivers
    ↓
if overtime is still necessary:
report it explicitly for dispatcher action
```

The objective therefore strongly minimizes **total overtime across the fleet**, rather than making overtime an automatic infeasibility.

### Important open-data limitation

CWWP2 is a highway travel-time feed, not a TomTom-style current traffic-flow layer for every local street.

Therefore:

- local/arterial roads mainly use the Irvine official baseline speed
- current dynamic traffic is strongest on mapped Caltrans highway corridors
- what can be achieved with open data depends on the city, county, state, or country

This limitation is part of the demonstration rather than something hidden.


## 23. Natural next extensions

1. Compare **07A1 TomTom vs 07B1 Open Data** using the same customers, orders, trucks, and driver shifts
2. Validate more CWWP2 endpoint / route fields and improve highway corridor matching
3. Add Caltrans PeMS historical traffic profiles
4. Add OCTA arterial traffic-volume or travel-time sources where appropriate
5. Add weekday × time-of-day historical traffic models
6. Move from a single current snapshot to a **time-dependent VRP**
7. Optimize driver-to-truck assignment instead of fixing it before the VRP
8. Add automatic driver shift swaps / relief-driver decisions
9. Add breaks, split shifts, and overtime wage rules
10. Add loading-order, pallet, fragile-item, and irregular-shape constraints

For **07B1**, the technical message is:

> **The same workforce-aware delivery optimization can run on open/public road data when suitable local GIS and transportation feeds exist — but the available real-time coverage depends strongly on the region.**
